In [1]:
#######################################################
# Armazena os pareceres e peditoes de todos os pedidos que estão na carga e atualiza anterioridades_desc e anterioridades
# na medida em que a tabela de carga foi atualizada há novos registros que precisam ter as petições anonimizadas e análises de IA atualizadas
# 1. CRIA NOVOS REGISTROS EM ANTERIORIDADES_DESC E ATUALIZA CAMPO DESCRICAO COM PARECER ANONIMIZADO E LIMPO
# aplique os INSERTs obtidos em insert_anterioridades_desc na tabela anterioridades_desc local do computador e no hostgator
# apos as inserções em anterioridades_desc confira 
# todos pedidos na carga tem entrada em anterioridade_desc: 
# SELECT * FROM `carga` WHERE  numero in (select numero from arquivados where despacho='12.2' and anulado=0) and numero not in (select numero from anterioridades_desc)
# todos pedidos na carga tem relatório completo em descricao
# SELECT * FROM `carga` WHERE numero in (select numero from arquivados where despacho='12.2' and anulado=0) and numero not in (select numero from anterioridades_desc where descricao like '%SERVIÇO PÚBLICO%')

# 2. SALVA PARECERES ANOMIZADOS DA PRIMEIRA INSTÂNCIA NA PASTA PARECERES
# 3. SALVA PETIÇÕES 214
# 4. LIMPA MANUALMENTE TXT PROCURANDO CPF, PAGAMENTO, HYAIP.COM, DBBA.COM, VCPI.COM, WWW.DANIEL,  GRUENBAUM.COM
# 5. TESTA CONTRARRAZOES insert ignore into anterioridades_ocorrencias
# 6. PREENCHA CAMPOS DE ANTERIORIDADES_DESC 
# https://cientistaspatentes.com.br/central/control.php?action=190&op=14
# https://cientistaspatentes.com.br/central/control.php?action=190&op=13
# https://cientistaspatentes.com.br/central/control.php?action=190&op=12
# https://cientistaspatentes.com.br/central/control.php?action=190&op=15
# https://cientistaspatentes.com.br/central/control.php?action=190&op=16

import os, re
import json
import requests
import pandas as pd

In [2]:
import json
import requests

def conectar_siscap(url,return_json=False):
    headers = {
        "Accept": "application/json",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    response = requests.get(url,headers=headers,verify=False)
    if response.status_code == 200:
        if return_json:
            try:
                data = response.json()
                json_data = json.dumps(data, indent=4)
                return(json_data)
            except Exception as e:
                return(f"Erro: {e}")
        else:
            return response.text
    else:
        return(f"Erro: {response.status_code}")


In [3]:


import mysql.connector
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

numero = "PI0808715"
# atualiza no localhost as tres tabelas: carga, anterioridades e anterioridades_desc
comando = f"SELECT * FROM arquivados WHERE numero='{numero}'"
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)
print(resultado)

[(1011532, '1.1', 'PI0808715', datetime.date(2011, 8, 9), 'dialp', 0, 0), (1475660, '1.3', 'PI0808715', datetime.date(2014, 8, 12), 'dialp', 0, 0), (1501853, '6.6', 'PI0808715', datetime.date(2014, 9, 16), 'dialp', 0, 0), (2788968, '7.1', 'PI0808715', datetime.date(2017, 4, 18), 'dialp', 0, 1), (2790019, '15.11', 'PI0808715', datetime.date(2017, 4, 18), 'dialp', 0, 0), (2956330, '9.2', 'PI0808715', datetime.date(2017, 9, 19), 'dialp', 0, 2), (3015281, '12.2', 'PI0808715', datetime.date(2017, 12, 19), 'dialp', 0, 0)]


In [4]:
import re
def limpa_texto_relatorio(numero, texto_relatorio):
    #padrao = r"(SERVIÇO PÚBLICO FEDERAL.*)"  # agora inclui o marcador
    padrao = r"(RELATÓRIO DE EXAME TÉCNICO.*)"  # agora inclui o marcador
    match = re.search(padrao, texto_relatorio, re.DOTALL)
    if match:
        texto_relatorio = match.group(1).strip()

    texto_relatorio = """    
                                      SERVIÇO PÚBLICO FEDERAL
                                       MINISTÉRIO DA ECONOMIA
                           INSTITUTO NACIONAL DA PROPRIEDADE INDUSTRIAL""" + texto_relatorio
    texto_relatorio = texto_relatorio.replace('"', '')
    texto_relatorio = texto_relatorio.replace("'", '')
    texto_relatorio = texto_relatorio.replace('\n\n', '\n').replace('\r\r', '\r')
    #texto_relatorio = re.sub(r'[\x00-\x1F\x7F-\x9F]', '', texto_relatorio)
    texto_relatorio = re.sub(r'[\x00-\x09\x0B\x0C\x0E-\x1F\x7F-\x9F]', '', texto_relatorio)
    texto_relatorio = texto_relatorio.replace('MODELO_SISCAP','')
    texto_relatorio = texto_relatorio.replace('Código:5975ce1e23737deeb22c88e902a79153versão1.3 19/04/12','')
    texto_relatorio = texto_relatorio.replace('Código:5975ce1e23737deeb22c88e902a79153versão1.1 27/01/12','')
    #texto_relatorio = ' '.join(texto_relatorio.split())
    #texto_relatorio = texto_relatorio.replace('  ', ' ')

    linhas = texto_relatorio.splitlines()
    linhas_processadas = []
    padrao = rf"(?:Página\s+)*"
    for linha in linhas:
        if linha.strip().startswith("N.° do Pedido:"):
            # mantém a linha intacta
            linhas_processadas.append(linha)
        else:
            # aplica limpeza
            linha_limpa = re.sub(padrao, "", linha)
            linha_limpa = linha_limpa.replace('BR' + numero + '-', '')
            linha_limpa = linha_limpa.replace('BR' + numero, '')
            linha_limpa = linha_limpa.replace(numero, '')
            linhas_processadas.append(linha_limpa)
    texto_relatorio = "\n".join(linhas_processadas)

    #texto_relatorio = re.sub(r"Mat\.\s*Nº.*", '', texto_relatorio, flags=re.MULTILINE) #  Mat. N.º
    #texto_relatorio = re.sub(r"Mat\.\s*nº.*", '', texto_relatorio, flags=re.MULTILINE) #  Mat. N.º
    texto_relatorio = re.sub(r'^.*(Mat\.\s*[Nn][º°]|Deleg\.\s*Comp\.).*$\n?', '', texto_relatorio, flags=re.MULTILINE) # elimina apenas alinha em que ocorre 
    #texto_relatorio = re.sub(r'^.*Deleg\. Comp\..*$\n?', '', texto_relatorio, flags=re.MULTILINE) # elimina apenas alinha em que ocorre
    texto_relatorio = re.sub(r'-+', '-', texto_relatorio) # elimine sequencias de ----
    texto_relatorio = re.sub(r'\n{3,}', '\n', texto_relatorio) # elimine pula linha dupla, triplas, ou mais
    linhas = texto_relatorio.splitlines()
    linhas_filtradas = []
    for linha in linhas:
        if re.match(r'^\s*\d+\s*$', linha):
            continue  # pula linha com número solto
        linhas_filtradas.append(linha)
    texto_relatorio = "\n".join(linhas_filtradas)
    texto_relatorio = re.sub(r'\s*(Quadro\s+[1-5])',r'\n\n\1',texto_relatorio)
    #print(texto_relatorio)
    #break
    return texto_relatorio

In [5]:
# 1. CRIA NOVOS REGISTROS EM ANTERIORIDADES_DESC E ATUALIZA CAMPO DESCRICAO COM PARECER ANONIMIZADO E LIMPO.

# select * from CEPIT_SISCAP.SISCAP_CARGA where numero in (select numero from CEPIT_SISCAP.SISCAP_arquivados WHERE despacho in ('12.2') and anulado=0)
# salve em carga .csv e carregue isso no phpmyadmin local
# zere o arquivo descricao.sql 

import mysql.connector
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

comando = f"select * from carga where numero<>'NUMERO' and numero not in (select numero from anterioridades_desc) and numero in (select numero from arquivados where despacho='12.2')"
#comando = f"select * from carga where numero<>'NUMERO' and numero not in (select numero from anterioridades_desc)"
#comando = f"SELECT * FROM `carga` WHERE numero in (select numero from arquivados where despacho='12.2' and anulado=0) and numero not in (select numero from anterioridades_desc where descricao like '%SERVIÇO PÚBLICO%')"
#comando = f"SELECT * FROM `carga` WHERE numero in (select numero from arquivados where despacho='12.2' and anulado=0)"
#comando = f"select * from anterioridades_desc where numero<>'NUMERO' and numero not in (select numero from carga)"
# teste se existe algum pedido na carga com 12.2 que ainda não tenha registro em anterioridades_desc:
# f"SELECT * FROM `carga` WHERE numero<>'NUMERO' and numero not in (select numero from anterioridades_desc) and numero in (select numero from arquivados where despacho='12.2');"
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)
#print(resultado)
lista = df.values.tolist()
if df.shape[1] > 1:
    lista = df.iloc[:, 0].tolist()
else:
    lista = []
lista.insert(0, 'numero')

#lista = ['numero','112013030358']
print(lista)

['numero']


In [27]:
#atualiza campo descricao da tabela anterioridades_desc
#certifique-se de estar conectado no SISCAP via VPN

data = {}
data["patents"] = lista
with open("descricao.sql", "a", encoding="utf-8") as f:
    for i in range(1, len(data["patents"])):
        numero = data["patents"][i] 
        query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM pedido where (decisao='ciencia de parecer' or decisao='indeferimento') and numero='{numero}'" + ' order by rpi desc"'
        url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
        print(url)
        try:
            json_data = conectar_siscap(url,return_json=True)
            data1 = json.loads(json_data)
            if data1.get("patents"):
                codigo = data1["patents"][0]["codigo"]
                divisao = data1["patents"][0]["divisao"]
            else:
                continue   
            
            url = f"https://siscap.inpi.gov.br/adm/pareceres/{divisao}/{numero}{codigo}.txt"
            print(url)
            texto_relatorio = conectar_siscap(url,return_json=False)
            texto_relatorio = limpa_texto_relatorio(numero, texto_relatorio)
            #print(texto_relatorio)

            sql_resumo = f"INSERT IGNORE INTO anterioridades_desc (id,numero,descricao) VALUES (null, '{numero}', '{texto_relatorio}');"
            #sql_resumo = f"UPDATE anterioridades_desc SET descricao='{texto_relatorio}' WHERE numero='{numero}';"
            #print(sql_resumo)
            f.write(sql_resumo + "\n")
            #if (i==5): break
            #print('###############')
        except Exception as e:
            print(f"Erro: {e}")
            continue

In [28]:
# acerta o campo descricao de quem nao tem o conteudo completo gravado neste campo
# para garantir que esta query gere zero resultados, ou seja, todos pedidos da carga estao com campo descricao com o parecer TXT 9.2 inteiro
# SELECT * FROM `carga` WHERE numero in (select numero from arquivados where despacho='12.2' and anulado=0) and numero not in (select numero from anterioridades_desc where descricao like '%SERVIÇO PÚBLICO%')
# se já estiver zerado esta query, não precisa rodar

data = {}
data["patents"] = lista
with open("descricao.sql", "a", encoding="utf-8") as f:
    for i in range(1, len(data["patents"])):
        numero = data["patents"][i] 
        if (numero=='numero'): continue
        query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM pedido where (decisao='indeferimento' or decisao='ciencia de parecer') and numero='{numero}'" + ' order by rpi desc"'
        url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
        print(url)
        try:
            json_data = conectar_siscap(url,return_json=True)
            data1 = json.loads(json_data)
            if data1.get("patents"):
                codigo = data1["patents"][0]["codigo"]
                divisao = data1["patents"][0]["divisao"]
            else:
                continue   
            
            caminho_do_arquivo = f"pareceres/{numero}{codigo}.txt"
            if os.path.exists(caminho_do_arquivo):
                with open(caminho_do_arquivo, 'r', encoding='utf-8') as arquivo:
                    texto_relatorio = arquivo.read()
            else:
                print(f"Arquivo não encontrado: {caminho_do_arquivo}")
                texto_relatorio = None  # ou "" dependendo do seu uso
                
            if texto_relatorio is not None:
                texto_relatorio = limpa_texto_relatorio(numero, texto_relatorio)
                #print(texto_relatorio)
                #break
                
                sql_resumo = f"UPDATE anterioridades_desc SET descricao='{texto_relatorio}' WHERE numero='{numero}';"
                print(sql_resumo)
                f.write(sql_resumo + "\n")
                #if (i==5): break
                #print('###############')
        except Exception as e:
            print(f"Erro: {e}")
            continue

In [ ]:
# testei fazer direto no sinergias mas demora muito !!
from urllib.parse import quote

query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM carga" + '"'
url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
json_data = conectar_siscap(url,return_json=True)
if isinstance(json_data, str):
    json_data = json.loads(json_data)

lista = []
for i in range(1, len(json_data["patents"])):
    numero = json_data["patents"][i]['numero'] # para ler a lista de numeros especificos
    query = f'"mysql_query":" * FROM arquivados where despacho=\'12.2\' and anulado=0 and numero=\'{numero}\'"'
    query_encoded = quote(query)
    url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query_encoded}"
    # print(url)
    json_data1 = conectar_siscap(url,return_json=True)
    if isinstance(json_data1, str) and "Erro" in json_data1:
        continue
    data1 = json.loads(json_data1)
    lista.append(numero)

print(lista)

In [6]:

# 2. SALVA PARECERES ANOMIZADOS DA PRIMEIRA INSTÂNCIA NA PASTA PARECERES

TIPO_CHOICES = [
        ('6.1', 'exigência (6.1)'),
        ('exigencia', 'exigência (6.1)'),
        ('7.1', 'ciência de parecer (7.1)'),
        ('ciencia de parecer', 'ciência de parecer (7.1)'),
        ('9.2', 'indeferimento (9.2)'),
        ('indeferimento', 'indeferimento (9.2)'),
        ('recurso provido-reforma 100.1', 'recurso provido-reforma (100.1)'),
        ('recurso provido-devolucao 100.2', 'recurso provido-devolução (100.2)'),
        ('recurso provido', 'recurso provido (100)'),
        ('recurso provido anvisa', 'recurso provido (100)'),
        ('recurso negado', 'recurso negado (111)'),
        ('recurso exigencia', 'recurso exigência (121)'),
        ('recurso ciencia', 'recurso ciência (121)'),
        ('recurso exigencia 121', 'recurso exigência (121)'),
        ('130', 'recurso prejudicado (130)'),
        ('200', 'depósito (200)'),
        ('202', 'publicação antecipada (202)'),
        ('203', 'pedido de exame de invenção (203)'),
        ('204', 'pedido de exame de modelo de utilidade (204)'),
        ('205', 'pedido de exame de certificado de adição (205)'),
        ('207', 'cumprimento exigência (207)'),
        ('210', 'subsídios ao exame (210)'),
        ('214', 'recurso administrativo (214)'),
        ('215', 'nulidade administrativa (215)'),
        ('216', 'contestação à nulidade (216)'),
        ('260', 'outras petições (260)'),
        ('272', 'manifestação sobre parecer técnico de recurso (272)'),
        ('280', 'cumprimento exigência de recurso (280)'),
        ('281', 'manifestação em primeira instância (281)'),
        ('282', 'manifestação sobre nulidade (282)'),
        ('284', 'pedido de exame de invenção via PCT com ISA/IPEA BR (284)'),
        ('285', 'pedido de exame de modelo de utilidade com ISA/IPEA BR (285)'),
        ('295', 'contrarrazões ao recurso (295)'),
        ('296', 'cumprimento exigência formal (296)'),
]

decisao = 'indeferimento'
resultado = [v for k, v in TIPO_CHOICES if k == decisao][0]
print(resultado)

def converter_data(data_iso):
    ano, mes, dia = data_iso.split("-")
    return f"{dia}/{mes}/{ano}"

indeferimento (9.2)


In [ ]:
# leitura dos pareceres técnicos de exigencia, ciencia e indeferimento do SISCAP e salva TXT localmente
# faz a leitura dos pareceres no SISCAP. Não faz uso de LLM. certifique-se de ter a tabela anterioridades_desc atualizada

query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM carga" + '"'
#query = '"' + "mysql_query" + '"' ":" + '"' + f" numero FROM anterioridades_desc" + '"'
url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
print(url)
json_data = conectar_siscap(url,return_json=True)
    
#json_data='{"patents": [{"numero":"PI0905487","prioridade":"BR","instancia":"2 exame","decisao":"indeferimento","prioritario":"0","cc1":"4","anulado":"0","codigo":"1340921","rpi":"2020-12-29","divisao":"dicel","etapa":"2"}]}'
json_data = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', json_data)
json_data = json_data.replace('\r', '')
#print(json_data)
#data = json.loads(json_data) # le os dados do sinergias
data["patents"] = lista # le os dados da lista do xampp localmente

for patent in data.get("patents", []):
    #numero = patent.get("numero")
    numero = patent
    if not numero or numero == "NUMERO":
        continue
        
    query_pedido = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM pedido where decisao in ('indeferimento','9.2','exigencia','ciencia de parecer') and rpi is not null and numero='{numero}'" + '"'
    url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query_pedido}"
    print(url)

    codigo = None
    divisao = None
    try:
        json_data = conectar_siscap(url,return_json=True)
    except:
        pass

    if not json_data:
        continue

    if json_data:
        try:
            #json_data='{"patents": [{"numero":"PI0905487","prioridade":"BR","instancia":"2 exame","decisao":"indeferimento","prioritario":"0","cc1":"4","anulado":"0","codigo":"1340921","rpi":"2020-12-29","divisao":"dicel","etapa":"2"}]}'
            json_data = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', json_data)
            json_data = json_data.replace('\r', '')
            data_pedido = json.loads(json_data)
            #patents = data_pedido.get("patents", [])

            for i in range(0, len(data_pedido["patents"])):
                #if i==2: break
                numero = data_pedido["patents"][i]["numero"]
                codigo = data_pedido["patents"][i]["codigo"]
                divisao = data_pedido["patents"][i]["divisao"]
                decisao = data_pedido["patents"][i]["decisao"]
                data_rpi = data_pedido["patents"][i]["rpi"]
                nova_data = converter_data(data_rpi) if data_rpi else ""

                if not codigo or not divisao or divisao == "sanot":
                    continue

                url = f"https://siscap.inpi.gov.br/adm/pareceres/{divisao}/{numero}{codigo}.txt"
                print(url)
            
                try:
                    texto_relatorio = conectar_siscap(url, return_json=False)
                except:
                    continue
                    
                if not texto_relatorio or len(texto_relatorio.strip()) < 50:
                    continue

                texto_relatorio = limpa_texto_relatorio(numero, texto_relatorio)
                caminho_do_arquivo = f"pareceres/{numero}{codigo}.txt"
                # os.makedirs(os.path.dirname(caminho_do_arquivo), exist_ok=True)

                resultado = [v for k, v in TIPO_CHOICES if k == decisao][0]
                output = numero + '\n' + "Parecer Técnico " + resultado + '\n'
                output = output + "Data de publicação na RPI: " + nova_data + '\n\n'
                texto_relatorio = output + texto_relatorio
                texto_relatorio = re.sub(r'(Pesquisador).*', r'\1', texto_relatorio, flags=re.S)
                #print(texto_relatorio)
                
                if not os.path.exists(caminho_do_arquivo):
                    #os.makedirs(os.path.dirname(caminho_do_arquivo), exist_ok=True)
                    print(f"Arquivo novo criado: {caminho_do_arquivo}")
                    with open(caminho_do_arquivo, "w", encoding="utf-8") as arquivo:
                        arquivo.write(texto_relatorio)
                else:
                    print(f"Arquivo já existe, não sobrescrito: {caminho_do_arquivo}") 
                    #with open(caminho_do_arquivo, "w", encoding="utf-8") as arquivo:
                    #    arquivo.write(texto_relatorio)

                    
        except json.JSONDecodeError:
            pass  # JSON inválido → segue o fluxo sem abortar
            

In [7]:
import re
import hashlib
from collections import defaultdict

class DataAnonymizer:
    def __init__(self):
        # token -> valor original
        self.token_map = {}

        # valor original -> token (garante determinismo)
        self.reverse_map = {}

        # contadores por tipo
        self.token_counter = defaultdict(int)

    # ==============================
    # GERAÇÃO DE TOKEN
    # ==============================
    def _generate_token(self, tipo, valor):
        if not valor:
            return valor

        valor = valor.strip()

        # Determinístico: mesma string → mesmo token
        if valor in self.reverse_map:
            return self.reverse_map[valor]

        self.token_counter[tipo] += 1
        token = f"[{tipo}_{self.token_counter[tipo]}]"

        self.token_map[token] = valor
        self.reverse_map[valor] = token

        return token

    # ==============================
    # REMOÇÃO DE CABEÇALHOS
    # ==============================
    def remover_cabecalhos(self, texto):
        if not isinstance(texto, str):
            return ""
        padroes_remover = [
            r"Assinado digitalmente por.*",
            r"Documento assinado eletronicamente.*",
            r"Protocolo:\s*\d+",
            r"URL para download:.*",
            r"Hash de autenticação:.*",
        ]

        for padrao in padroes_remover:
            texto = re.sub(padrao, "", texto, flags=re.IGNORECASE)

        return texto

    # ==============================
    # ANONIMIZAÇÃO DE CPFs
    # ==============================
    def anonymize_cpfs(self, texto):
        if not isinstance(texto, str):
            return ""
        regex_cpf = r"\b\d{3}\.\d{3}\.\d{3}-\d{2}\b"

        def substituir(match):
            cpf = match.group()
            return self._generate_token("CPF", cpf)

        return re.sub(regex_cpf, substituir, texto)

    # ==============================
    # ANONIMIZAÇÃO DE CNPJ
    # ==============================
    def anonymize_cnpj(self, texto):
        if not isinstance(texto, str):
            return ""
        regex_cnpj = r"\b\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2}\b"

        def substituir(match):
            cnpj = match.group()
            return self._generate_token("CNPJ", cnpj)

        return re.sub(regex_cnpj, substituir, texto)

    # ==============================
    # ANONIMIZAÇÃO DE PROCESSOS (9 dígitos)
    # ==============================
    def anonymize_processos(self, texto):
        if not isinstance(texto, str):
            return ""
        regex_processo = r"\b\d{9}\b"

        def substituir(match):
            processo = match.group()
            return self._generate_token("PROCESSO", processo)

        return re.sub(regex_processo, substituir, texto)

    # ==============================
    # ANONIMIZAÇÃO DE EMAIL
    # ==============================
    def anonymize_emails(self, texto):
        if not isinstance(texto, str):
            return ""
        regex_email = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"

        def substituir(match):
            email = match.group()
            return self._generate_token("EMAIL", email)

        return re.sub(regex_email, substituir, texto)

    # ==============================
    # ANONIMIZAÇÃO DE NOMES SIMPLES (heurística)
    # ==============================
    def anonymize_nomes_maiusculos(self, texto):
        if not isinstance(texto, str):
            return ""
        # Heurística: nomes em caixa alta com pelo menos 2 palavras
        regex_nome = r"\b([A-ZÁÉÍÓÚÂÊÔÃÕÇ]{2,}(?:\s+[A-ZÁÉÍÓÚÂÊÔÃÕÇ]{2,})+)\b"

        def substituir(match):
            nome = match.group(1)
            return self._generate_token("PESSOA_NATURAL", nome)

        return re.sub(regex_nome, substituir, texto)


    # ==============================
    # DOCUMENTOS GENÉRICOS DE IDENTIFICAÇÃO
    # ==============================
    def anonymize_documentos_identificacao(self, texto):
        if not isinstance(texto, str):
            return ""
        regex = r"(CPF\/CNPJ|CPF|CNPJ)\s*:\s*([A-Z0-9\-\.\/]+)"
    
        def substituir(match):
            rotulo = match.group(1)
            valor = match.group(2)
            token = self._generate_token("DOC_ID", valor)
            return f"{rotulo}: {token}"
    
        return re.sub(regex, substituir, texto, flags=re.IGNORECASE)
        
    # ==============================
    # IDENTIFICADORES INTERNACIONAIS
    # ==============================
    def anonymize_identificadores_internacionais(self, texto):
        if not isinstance(texto, str):
            return ""
        regex = r"\b[A-Z]{2}\d{6,}\b"
    
        def substituir(match):
            valor = match.group()
            return self._generate_token("REGISTRO_INT", valor)
    
        return re.sub(regex, substituir, texto)

    # ==============================
    # ENDEREÇO
    # ==============================
    def anonymize_endereco(self, texto):
        if not isinstance(texto, str):
            return ""
        regex = r"(Endere[cç]o\s*:\s*)(.+)"
    
        def substituir(match):
            prefixo = match.group(1)   # "Endereço: "
            valor = match.group(2).strip()
    
            token = self._generate_token("ENDERECO", valor)
    
            return f"{prefixo}{token}"
    
        return re.sub(regex, substituir, texto, flags=re.IGNORECASE)

    # ==============================
    # REMOVER A LINHA INTEIRA QUE TENHA CEP
    # ==============================
    def anonymize_remover_linhas_com_cep(self, texto):
        if not isinstance(texto, str):
            return ""
        linhas = texto.splitlines()
    
        linhas_filtradas = [
            linha for linha in linhas
            if not re.search(r'\bCEP\b', linha, flags=re.IGNORECASE)
        ]
    
        return "\n".join(linhas_filtradas)
    
    # ==============================
    # REMOVER CABEÇALHO DAS PÁGINAS DA PETIÇÃO
    # ==============================
    def anonymize_remover_cabecalhos_pagina(self, texto):
        if not isinstance(texto, str):
            return ""
        linhas = texto.splitlines()
    
        linhas_filtradas = [
            linha for linha in linhas
            if not re.search(r'^\s*Peticao\b', linha, flags=re.IGNORECASE)
        ]
    
        return "\n".join(linhas_filtradas)

    # ==============================
    # TELEFONE
    # ==============================
    def anonymize_telefone(self, texto):
        if not isinstance(texto, str):
            return ""
        regex = r"(Telefone|Fax\s*:\s*)(.+)"
        regex = r"\b(Fone\/Fax|Fone|Telefone|Tel\.?|Fax)\b\s*:?\s*([\(\d][\d\.\-\)\s]+)"
    
        def substituir(match):
            prefixo = match.group(1)   # "Endereço: "
            valor = match.group(2).strip()
    
            token = self._generate_token("TELEFONE", valor)
    
            return f"{prefixo}{token}"
    
        return re.sub(regex, substituir, texto, flags=re.IGNORECASE)

    # ==============================
    # NOMES ROTULADOS
    # ==============================
    def anonymize_nomes_rotulados(self, texto):
        if not isinstance(texto, str):
            return ""
        regex = r"(Requerente|Técnico|Inventor|Procurador)\s*:\s*([A-ZÁÉÍÓÚÂÊÔÃÕÇ\s]+)"
    
        def substituir(match):
            rotulo = match.group(1)
            nome = match.group(2).strip()
            token = self._generate_token("PESSOA_NATURAL", nome)
            return f"{rotulo}: {token}"
    
        return re.sub(regex, substituir, texto)
        
    # ==============================
    # PIPELINE COMPLETO
    # ==============================
    def anonymize_texto(self, texto):
        texto = self.remover_cabecalhos(texto)
        texto = self.anonymize_cpfs(texto)
        texto = self.anonymize_cnpj(texto)
        texto = self.anonymize_emails(texto)
        texto = self.anonymize_processos(texto)
        #texto = self.anonymize_nomes_maiusculos(texto) # este critério estava retirando qualquer sequencia de maiusculos o que elimina título do pedido
        texto = self.anonymize_documentos_identificacao(texto)

        #texto = self.anonymize_identificadores_internacionais(texto)
        texto = self.anonymize_endereco(texto)
        texto = self.anonymize_telefone(texto)
        texto = self.anonymize_nomes_rotulados(texto)
        texto = self.anonymize_remover_linhas_com_cep(texto)
        texto = self.anonymize_remover_cabecalhos_pagina(texto)

        return texto

    
    # ==============================
    # DESTOKENIZAÇÃO
    # ==============================
    def deanonymize(self, texto):
        # Substitui tokens pelos valores originais
        for token, valor in self.token_map.items():
            texto = texto.replace(token, valor)
        return texto

    def iniciar_apos_recurso_207(self, texto):
        if not isinstance(texto, str):
            return ""
        padrao = r"(RESPOSTA\s+[AÀ]\s+EXIG[EÊ]NCIA\b|CUMPRIMENTO\s+DE\s+EXIG[EÊ]NCIA\b|RESPOSTA\s+[AÀ]\s+CI[EÊ]NCIA\b|MANIFESTA[ÇC][AÃ][O0]\s+(?:SOBRE\s+O|AO)\s+EXAME\b|E\s*S\s*C\s*L\s*A\s*R\s*E\s*C\s*I\s*M\s*E\s*N\s*T\s*O\s*S?|Excelent[ií]ssim[oa]\b|Ilustr[ií]ssim[oa]\s|Ilm[oa]\s+Senhor\s+|Ilm[oa]\s+Senhora\b)"
        match = re.search(padrao, texto, flags=re.IGNORECASE | re.DOTALL)
        if match:
            return texto[match.start():]
        else:
            return

    def iniciar_apos_recurso_210(self, texto):
        if not isinstance(texto, str):
            return ""
        padrao = r"(SUBS[ÍI]DIOS\s+(?:AO|PARA\s+O)\s+EXAME\s+T[ÉE]CNICO\b|RAZ[OÕ]ES\b|R\s*A\s*Z\s*[ÕO]\s*E\s*S|Excelent[ií]ssim[oa]\b|Ilustr[ií]ssim[oa]\s|Ilm[oa]\s+Senhor\s+|Ilm[oa]\s+Senhora\b)"
        match = re.search(padrao, texto, flags=re.IGNORECASE | re.DOTALL)
        if match:
            return texto[match.start():]
        else:
            return

    def iniciar_apos_recurso_260(self, texto):
        if not isinstance(texto, str):
            return ""
        padrao = r"(CONTRARRAZ[ÕO]ES\s+(?:AO|SOBRE\s+O)\s+RECURSO\b|CONTRARRAZ[ÕO]ES\s+(?:AO|SOBRE\s+O)\s+EXAME\b|APRESENTA[ÇC][ÃA]O\s+DE\s+CONTRARRAZ[ÕO]ES\b|E\s*S\s*C\s*L\s*A\s*R\s*E\s*C\s*I\s*M\s*E\s*N\s*T\s*O\s*S?)|RAZ[OÕ]ES\b|R\s*A\s*Z\s*[ÕO]\s*E\s*S"
        match = re.search(padrao, texto, flags=re.IGNORECASE | re.DOTALL)
        if match:
            return texto[match.start():]
        else:
            return

    def iniciar_apos_recurso_281(self, texto):
        if not isinstance(texto, str):
            return ""
        padrao = r"(Cumprimento\s+de\s+Exig[êe]ncia\b|Justificativa\s+de\s+Patente\b|MANIFESTA[CÇ][AÃ]O\s+(?:SOBRE\s+O|SOBRE|A|AO)\s+PARECER\b|MANIFESTA[CÇ][AÃ]O\s+(?:SOBRE|A|À)\s+CI[ÊE]NCIA\b|RAZ[OÕ]ES\b|R\s*A\s*Z\s*[ÕO]\s*E\s*S|E\s*S\s*C\s*L\s*A\s*R\s*E\s*C\s*I\s*M\s*E\s*N\s*T\s*O\s*S?|Excelent[ií]ssim[oa]\b|Ilustr[ií]ssim[oa]\s|Ilm[oa]\s+Senhor\s+|Ilm[oa]\s+Senhora\b)"
        match = re.search(padrao, texto, flags=re.IGNORECASE | re.DOTALL)
        if match:
            return texto[match.start():]
        else:
            return

    def iniciar_apos_recurso_214(self, texto):
        if not isinstance(texto, str):
            return ""
        padrao = r"(RECURSO\s+do\s+despacho\s+que\s+indeferiu\s+o\s+Pedido\s+de\s+Paten\s*te|Excelent[ií]ssimo\b|Ilustr[ií]ssimo\s+Senhor\b|Ilmo\s+Senhor\s+Presidente\b|recurso\s+ao\s+presidente\s+|apresentar\s+recurso\s+desta\s+decis[aã]o|Em\s+resposta\s+a\s*(?:o|ao)\s+Indeferi\s*-?\s*mento)"
        match = re.search(padrao, texto, flags=re.IGNORECASE | re.DOTALL)
        if match:
            return texto[match.start():]
        else:
            padrao = r"^\s*RECURSO\s+CONTRA\s+INDEFERIMENTO\s*$"
            match = re.search(padrao, texto, flags=re.IGNORECASE | re.MULTILINE)
            if match:
                return texto[match.start():]
            else:
                padrao = r"^\s*RECURSO\s+AO\s+INDEFERIMENTO\s*$"
                match = re.search(padrao, texto, flags=re.IGNORECASE | re.MULTILINE)
                if match:
                    return texto[match.start():]
                else:
                    padrao = r"^\s*INTERPOSICAO\s+DE\s+RECURSO\s+AO\s+INDEFERIMENTO\s*$"
                    match = re.search(padrao, texto, flags=re.IGNORECASE | re.MULTILINE)
                    if match:
                        return texto[match.start():]
                    else:
                        padrao = r"^\s*RECURSO\s+CONTRA\s+INDEFERIMENTO,\s*$"
                        match = re.search(padrao, texto, flags=re.IGNORECASE | re.MULTILINE)
                        if match:
                            return texto[match.start():]
                        else:
                            padrao = r"(RECURSO\s+ADMINISTRATIVO\s+|recurso\s+contra\s+o\s+indeferimento|recurso\s+ao\s+indeferimento|recurso\s+contra\s+decisao\s+de\s+indeferimento)"
                            match = re.search(padrao, texto, flags=re.IGNORECASE | re.DOTALL)
                            if match:
                                return texto[match.start():]
                            else:
                                padrao = r"(ilustrissimos\s+examinadores)"
                                match = re.search(padrao, texto, flags=re.IGNORECASE | re.DOTALL)
                                if match:
                                    return texto[match.start():]
                                else:
                                    padrao = r"(recurso\s+que\s+bastante\s+faz)"
                                    match = re.search(padrao, texto, flags=re.IGNORECASE | re.DOTALL)
                                    if match:
                                        return texto[match.start():]
                                    return ""

import re
import unicodedata

def normalizar(texto):
    texto = texto.replace('\xa0', ' ')  # remove non-breaking space
    texto = unicodedata.normalize('NFKD', texto)
    texto = texto.encode('ASCII', 'ignore').decode('ASCII')  # remove acentos
    #texto = re.sub(r'\s+', ' ', texto)  # colapsa múltiplos espaços/quebras
    return texto
        
documento = """
RECURSO do despacho que indeferiu o Pedido de Patente
Requerente: JOÃO SILVA OLIVEIRA CPF 123.456.789-00
Técnico: RICARDO FREDERICO NICOL
Processo anterior: 123456789
Email: joao@email.com
Assinado digitalmente por servidor INPI.
"""

anonymizer = DataAnonymizer()
documento = normalizar(documento)
#documento = anonymizer.iniciar_apos_recurso(documento)
texto_anon = anonymizer.anonymize_texto(documento)
print("=== DOCUMENTO ANONIMIZADO ===")
print(texto_anon)

=== DOCUMENTO ANONIMIZADO ===

RECURSO do despacho que indeferiu o Pedido de Patente
Requerente: [PESSOA_NATURAL_1][CPF_1]
Tecnico: RICARDO FREDERICO NICOL
Processo anterior: [PROCESSO_1]
Email: [EMAIL_1]


In [ ]:
import PyPDF2, os

arquivo = 'PI1013364_29409161939772965_207.pdf'

nome_sem_extensao = arquivo.replace('.pdf', '')
partes = nome_sem_extensao.split('_')
numero = partes[0]
numnossonumero = partes[1]
tipo = partes[2]

file_path = f"pareceres/peticoes/{numero}_{numnossonumero}_{tipo}.pdf"
print(file_path)
all_text = ''
if os.path.exists(file_path):
    with open(file_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            all_text += page.extract_text() or ""
else:
    print(f"Arquivo não encontrado: {file_path}")
    
def anonimizacao(texto, tipo):
    texto_anon = ''
    anonymizer = DataAnonymizer()
    documento_normalizado = normalizar(texto) # elimina quebras de linha e caracteres especiais
    documento_inicial = documento_normalizado
    if tipo=='214':
        documento_inicial = anonymizer.iniciar_apos_recurso_214(documento_normalizado) # busca cabeçalho de início
    if tipo=='207':
        documento_inicial = anonymizer.iniciar_apos_recurso_207(documento_normalizado) # busca cabeçalho de início
    if tipo=='210':
        documento_inicial = anonymizer.iniciar_apos_recurso_210(documento_normalizado) # busca cabeçalho de início
    if tipo=='260':
        documento_inicial = anonymizer.iniciar_apos_recurso_260(documento_normalizado) # busca cabeçalho de início
    if tipo=='281':
        documento_inicial = anonymizer.iniciar_apos_recurso_281(documento_normalizado) # busca cabeçalho de início
    texto_anon = anonymizer.anonymize_texto(documento_inicial) # anonimiza referencias
    #print("=== DOCUMENTO ANONIMIZADO ===")
    #print(texto_anon)
    return texto_anon

#print(all_text)
texto_anon = anonimizacao(all_text,tipo)
print(texto_anon)
if texto_anon == '':
    print("Texto inicial não identificado")

In [ ]:
# donwload das peticoes
# varios registros estao com cd_imagem=1
# numnossonumero=29409161953366073 cd_imagem=1
# para corrigir detectar no DBVISUALIZER os casos de 214
# select * from CEPIT_SISCAP.SISCAP_DESPACHOS_PAG where tipo_peticao='214' and cd_imagem>0
# salva em CSV e carrega no arquivo local do XAMPP na tabela despachos_pag
# delete FROM `despachos_pag`  where cd_imagem=0
# rode o algoritmo de correção de cd_imagem
# faça o import de updates.sql
# confira select * from despachos_pag where cd_imagem=0 and tipo_peticao='214';

import os
import json
import requests

#query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM carga" + '"'
query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM carga" + '"'
url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
print(url)

json_data = conectar_siscap(url,return_json=True)
json_data = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', json_data)
json_data = json_data.replace('\r', '')
data = json.loads(json_data)

#json_str = json.dumps(json_data)
#json_str = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', json_str)
#json_str = json_str.replace('\r', '')
#data = json.loads(json_str)

#print(json_data)

#tipo = '272' # Manifestação sobreparecer técnico proferido em grau de recurso
#tipo = '210' # Subsídios ao exame técnico
tipo = '260' # Outras petições
#tipo = '207' # Cumprimento de exigência
#tipo = '214' # Recurso
#tipo = '200' # Depósito inicial de pedido de patente
#tipo = '280' # Cumprimento de exigência em grau de recurso 
#tipo = '281' #  Manifestação sobre invenção, modelo de utilidade, certificado de adição de invenção em 1ª instância

##tipo = '202' # Publicação antecipada (dispensado de petição)
##tipo = '295'  # so tem 4 registros: Contrarrazões ao recurso
##tipo = '296'  # so tem 7 registros: Cumprimento de exigência formal

#data = {
#    "patents": [
#        {"numero": "102012030377"}
#    ]
#}

for patent in data.get("patents", []):
    numero = patent.get("numero")
    #divisao = patent.get("divisao")
    if not numero or numero == "NUMERO":
        continue

    # https://cientistaspatentes.com.br/apiphp/patents/query/?q="mysql_query":" * FROM despachos_pag WHERE numero='102013033208' and tipo_peticao='272'" 
    json_data = None
    query_pedido = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM despachos_pag where numero='{numero}' and tipo_peticao='{tipo}'"+'"' # 112015029938
    url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query_pedido}"
    print(url)

    numnossonumero = None
    cd_imagem = None
    try:
        json_data = conectar_siscap(url,return_json=True)
    except:
        pass

    if not json_data:
        continue

    #print(json_data)
  
    if json_data:
        try:
            json_data = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', json_data)
            json_data = json_data.replace('\r', '')
            data_pedido = json.loads(json_data)
            patents = data_pedido.get("patents", [])
            if not patents:
                continue
                
            for primeiro in patents:
                #numnossonumero = data_pedido["patents"][0]["numnossonumero"]
                #cd_imagem = int(data_pedido["patents"][0]["cd_imagem"])
                #data_peticao = data_pedido["patents"][0]["data_peticao"]
                #primeiro = patents[0]
                #if not isinstance(primeiro, dict):
                #    continue
                    
                numnossonumero = primeiro.get("numnossonumero")
                cd_imagem = int(primeiro.get("cd_imagem", 0))
                data_peticao = primeiro.get("data_peticao")
                nova_data = converter_data(data_peticao) if data_peticao else ""
                print (f"numnossonumero={numnossonumero} cd_imagem={cd_imagem} data_peticao={data_peticao}")

                if not cd_imagem or cd_imagem==1 or cd_imagem==0:
                    continue

                url = f"http://br00-aux.inpi.gov.br/webservice/retornaImagem.php?codigo={cd_imagem}"
                arquivo_saida = f"peticoes/{numero}_{numnossonumero}_{tipo}.pdf"

                try:
                    response = requests.get(url, stream=True, verify=False, timeout=30)
                
                    if response.status_code == 200:
                        if not os.path.exists(arquivo_saida):
                            with open(arquivo_saida, "wb") as f:
                                for chunk in response.iter_content(chunk_size=8192):
                                    if chunk:
                                        f.write(chunk)
                            print(f"Download concluído: {arquivo_saida}")
                            
                            all_text = ''
                            if os.path.exists(arquivo_saida):
                                with open(arquivo_saida, "rb") as file:
                                    reader = PyPDF2.PdfReader(file)
                                    for page in reader.pages:
                                        all_text += page.extract_text() or ""
                            else:
                                print(f"Arquivo não encontrado: {file_path}")
                        
                            texto_relatorio = anonimizacao(all_text,tipo)
                            
                            if not texto_relatorio or len(texto_relatorio.strip()) < 50:
                                texto_relatorio = anonimizacao(all_text,'completo') # 112012032204_29409161957989504_214 estava vazio !
                                
                            caminho_do_arquivo = f"peticoes/{numero}_{numnossonumero}_{tipo}.txt"
                            # os.makedirs(os.path.dirname(caminho_do_arquivo), exist_ok=True)

                            resultado = [v for k, v in TIPO_CHOICES if k == tipo][0]
                            output = numero + '\n' + "Petição " + resultado + '\n'
                            output = output + "Data da Petição: " + nova_data + '\n\n'
                            texto_relatorio = output + texto_relatorio
            
                            if not os.path.exists(caminho_do_arquivo):
                                #os.makedirs(os.path.dirname(caminho_do_arquivo), exist_ok=True)
                                print(f"Arquivo novo criado: {caminho_do_arquivo}")
                                with open(caminho_do_arquivo, "w", encoding="utf-8") as arquivo:
                                    arquivo.write(texto_relatorio)
                            else:
                                print(f"Arquivo já existe, não sobrescrito: {caminho_do_arquivo}")   
                        else:
                            print(f"O arquivo {arquivo_saida} já existe. Nada foi gravado.")
                    else:
                        print(f"Falha no download. HTTP {response.status_code}")
                
                except requests.exceptions.RequestException as e:
                    print(f"Erro na requisição: {e}")
   
                    
        except json.JSONDecodeError:
            pass  # JSON inválido → segue o fluxo sem abortar

In [ ]:

# Faça OCR dos PDF digital que tiveram TXT de menos de 512 bytes, ou seja, somente o cabeçalho padrão

import pytesseract
from pdf2image import convert_from_path
from PIL import Image
import pdfplumber
import subprocess

def ocr_pdf_imagem(pdf_path):
    # Definir o caminho do executável do Tesseract (necessário apenas no Windows)
    pytesseract.pytesseract.tesseract_cmd = "C:/Program Files/Tesseract-OCR/tesseract.exe"
    pytesseract.pytesseract.tesseract_cmd = "D:/Users/abrantes/AppData/Local/Programs/Tesseract-OCR/tesseract.exe"
    
    # Caminho para o arquivo PDF
    # instalação do poppler https://github.com/oschwartz10612/poppler-windows/releases
    poppler_path = r'D:\Users\abrantes\poppler-24.08.0\Library\bin'  
    
    # Converter PDF em uma lista de imagens (cada página do PDF será uma imagem)
    pages = convert_from_path(pdf_path, 300, poppler_path=poppler_path)  # 300 DPI para qualidade de imagem
    
    # Percorrer cada página e extrair o texto
    text = ""
    for page in pages:
        # Converter a imagem para texto usando o pytesseract (OCR)
        text += pytesseract.image_to_string(page, lang='por')  # 'lang' define o idioma, 'por' para português
    
    # Exibir o texto extraído
    return text

def extrair_com_ocr(pdf_path):

    pdf = pdfium.PdfDocument(pdf_path)
    texto = ""

    for page in pdf:

        bitmap = page.render(scale=400/72)
        image = bitmap.to_pil()

        t = pytesseract.image_to_string(image, lang="por")
        texto += t

    return texto

pdf_path = 'peticoes/PI1015975_29409161937794589_214.pdf'
pdf_path = 'peticoes/112015031143_0000221509591979_200.pdf'
pdf_path = 'peticoes/102012005921_0000221201668837_200.pdf'
pdf_path = 'peticoes/PI1004858_29409161923641327_281.pdf'
pdf_path = 'peticoes/102018068107_29409162311019704_281.pdf'

texto = ocr_pdf_imagem(pdf_path)
print(texto)

In [10]:
import pypdfium2 as pdfium
import pytesseract
from datetime import datetime

def verificar_frase(caminho_txt):
    try:
        with open(caminho_txt, 'r', encoding='utf-8') as arquivo:
            conteudo = arquivo.read().lower()  # deixa tudo minúsculo para evitar problemas
            if ("apresentação de procuração" in conteudo) or ("documento de cessão dos direitos" in conteudo) or ("correção de dados da prioridade/inventor" in conteudo):
                return True
            else:
                return False
    except FileNotFoundError:
        return True
    except Exception as e:
        return True
        
diretorio = "peticoes"
data_alvo = datetime.strptime("11/04/2026", "%d/%m/%Y").date()
for raiz, dirs, arquivos in os.walk(diretorio):
    for arquivo in arquivos:
        if arquivo.lower().endswith(".txt"):
            caminho_txt = os.path.join(raiz, arquivo)
            timestamp = os.path.getmtime(caminho_txt)
            data_arquivo = datetime.fromtimestamp(timestamp).date()
            if data_arquivo != data_alvo:
                continue
                
            tamanho = os.path.getsize(caminho_txt)  # teste o tamanho do TXT note que esse arquivo já tem o cabeçalho
            caminho_pdf = caminho_txt.replace('.txt','.pdf')
            if verificar_frase(caminho_txt):
                continue # esse arquivo já foi verificado e tem apenas a procuração, não precisa fazer OCR
            else:
                if tamanho < 512:  # 1 KB = 1024 bytes
                    m = re.search(r'_(\d+)\.txt$', caminho_txt)
                    if m:
                        tipo = m.group(1)
                        if tipo != '295':
                            match = re.search(r'\\(\d+)_', caminho_txt)
                            if match:
                                numero = match.group(1)
                            print(f"{caminho_txt} {caminho_pdf} - {tamanho} bytes {numero} {tipo}")
                            texto = extrair_com_ocr(caminho_pdf)       
                            texto_relatorio = anonimizacao(texto,tipo)
                            # resultado = [v for k, v in TIPO_CHOICES if k == tipo][0]
                            # output = numero + '\n' + "Petição " + resultado + '\n'
                            # output = output + "Data da Petição: " + nova_data + '\n\n'
                            # texto_relatorio = output + texto_relatorio
                            print(f"Arquivo novo criado {tipo}: {caminho_txt}")
                            with open(caminho_txt, "a", encoding="utf-8") as arquivo: # note que ele faz append
                                arquivo.write(texto_relatorio)

In [11]:
import re

import os

def remover_recibo_final(texto, limite_linhas=150):
    """
    Remove o bloco 'RECIBO DO SACADO' apenas se ele estiver no final do arquivo.
    Retorna (texto_modificado, houve_corte)
    """

    linhas = texto.splitlines()

    for i, linha in enumerate(linhas):
        if "RECIBO DO SACADO" in linha.upper():

            linhas_restantes = len(linhas) - i

            if linhas_restantes <= limite_linhas:
                return "\n".join(linhas[:i]), True

    return texto, False

def remover_instrucoes_final(texto, limite_linhas=150):
    """
    Remove o bloco 'A data de vencimento nao prevalece sobre o prazo legal' apenas se ele estiver no final do arquivo.
    Retorna (texto_modificado, houve_corte)
    """

    linhas = texto.splitlines()

    for i, linha in enumerate(linhas):
        if "PREVALECE SOBRE O PRAZO LEGAL. O PAGAMENTO DEVE SER EFETUADO" in linha.upper():

            linhas_restantes = len(linhas) - i

            if linhas_restantes <= limite_linhas:
                return "\n".join(linhas[:i]), True

    return texto, False

def remover_procuracao_final(texto, limite_linhas=150):
    """
    Remove o bloco 'PROCURACAO' apenas se ele estiver no final do arquivo.
    Retorna (texto_modificado, houve_corte)
    """

    linhas = texto.splitlines()

    for i, linha in enumerate(linhas):
        if "PROCURACAO" in linha.upper() or "P R O C U R A C A O" in linha.upper() or "P R O C U R A Ç Ã O" in linha.upper() or "PROCURAÇÃO" in linha.upper():

            linhas_restantes = len(linhas) - i

            if linhas_restantes <= limite_linhas:
                return "\n".join(linhas[:i]), True

    return texto, False

def remover_linhas_vazias_multiplas(texto):
    # Substitui 2 ou mais quebras de linha por apenas duas (\n\n = 1 linha em branco)
    return re.sub(r'\n\s*\n+', '\n\n', texto)

def remover_numeros_perdidos_na_linha(texto_relatorio):
    texto_relatorio = re.sub(r'-+', '-', texto_relatorio) # elimine sequencias de ----
    texto_relatorio = re.sub(r'\n{3,}', '\n', texto_relatorio) # elimine pula linha dupla, triplas, ou mais
    linhas = texto_relatorio.splitlines()
    linhas_filtradas = []
    i = 0 
    for linha in linhas:
        if i>0:
            if re.match(r'^\s*\d+\s*$', linha):
                continue  # pula linha com número solto
        linhas_filtradas.append(linha)
        i = i + 1
    texto_relatorio = "\n".join(linhas_filtradas)

    padrao = r'^\s*\d+\s*/\s*\d+\s*$'
    linhas_filtradas = [
        linha for linha in texto_relatorio.splitlines()
        if not re.match(padrao, linha)
    ]
    
    return '\n'.join(linhas_filtradas)

def processar_txt(entrada, saida):

    with open(entrada, "r", encoding="utf-8") as f:
        texto = f.read()

    texto = remover_numeros_perdidos_na_linha(texto)
    texto_limpo, houve_corte = remover_recibo_final(texto)
    if not houve_corte:
        print("Pulando arquivo (recibo não está no final):", entrada)
        texto_limpo, houve_corte = remover_instrucoes_final(texto)
        if not houve_corte:
            print("Pulando arquivo (instruções não está no final):", entrada)
            texto_limpo, houve_corte = remover_procuracao_final(texto)
            if not houve_corte:
                print("Pulando arquivo (procuração não está no final):", entrada)
                texto_limpo = remover_linhas_vazias_multiplas(texto_limpo)
                with open(saida, "w", encoding="utf-8") as f:
                    f.write(texto_limpo)
                return
                
    texto_limpo = remover_linhas_vazias_multiplas(texto_limpo)
    with open(saida, "w", encoding="utf-8") as f:
        f.write(texto_limpo)

    print("Arquivo criado:", saida)

def corrigir_arquivo(arquivo_entrada, arquivo_saida):
    nome_arquivo = os.path.basename(arquivo_entrada)
    
    # Extrai o número antes do primeiro "_"
    match = re.match(r'^(\d+)_', nome_arquivo)
    
    if not match:
        return  # não tem padrão esperado
    
    numero_pedido = match.group(1)
    
    # Garante que começa com dígito (na prática já garantido pelo \d+)
    if not re.match(r'^[0-9]', numero_pedido):
        return

    # Lê o conteúdo do arquivo
    with open(arquivo_entrada, 'r', encoding='utf-8') as f:
        linhas = f.readlines()
        
    # Verifica se já existe na primeira linha
    if linhas and linhas[0].strip() == numero_pedido:
        return  # já está correto

    # Insere o número como primeira linha
    novas_linhas = [numero_pedido + '\n'] + linhas

    # Salva o arquivo (sobrescreve ou salva em outro caminho)
    with open(arquivo_saida, 'w', encoding='utf-8') as f:
        f.writelines(novas_linhas)

    print("Arquivo criado:", arquivo_saida)

arquivo_entrada = "peticoes/102013033387_29409161940566800_214.txt"  # RECIBO DO SACADO AO FINAL
arquivo_entrada = "peticoes/102012004873_29409161938119486_281.txt"  # INSTRUCOES AO FINAL
arquivo_entrada = "peticoes/102012021084_29409161946481776_281.txt"  # PROCURACAO AO FINAL
arquivo_entrada = "peticoes/PI1103116_29409161929427670_214.txt" # com vários pula linha
arquivo_entrada = "peticoes/102012005921_0000221201668837_200.txt" # com várias linhas com número de página sozinho perdido

diretorio = os.path.dirname(arquivo_entrada)
nome_arquivo = os.path.basename(arquivo_entrada)
#arquivo_saida = os.path.join(diretorio, "new_" + nome_arquivo)
arquivo_saida = arquivo_entrada
processar_txt(arquivo_entrada, arquivo_saida)
#corrigir_arquivo(arquivo_entrada, arquivo_saida)

Pulando arquivo (recibo não está no final): peticoes/102012005921_0000221201668837_200.txt
Pulando arquivo (instruções não está no final): peticoes/102012005921_0000221201668837_200.txt
Pulando arquivo (procuração não está no final): peticoes/102012005921_0000221201668837_200.txt


In [12]:
diretorio = "peticoes"
data_alvo = datetime.strptime("11/04/2026", "%d/%m/%Y").date()
for nome_arquivo in os.listdir(diretorio):
    if nome_arquivo.lower().endswith(".txt"):
        arquivo_entrada = os.path.join(diretorio, nome_arquivo)
        arquivo_saida = arquivo_entrada
        timestamp = os.path.getmtime(arquivo_entrada)
        data_arquivo = datetime.fromtimestamp(timestamp).date()
        if data_arquivo != data_alvo:
            continue
            
        processar_txt(arquivo_entrada, arquivo_saida)
        #corrigir_arquivo(arquivo_entrada, arquivo_saida)

Pulando arquivo (recibo não está no final): peticoes\102012005921_0000221201668837_200.txt
Pulando arquivo (instruções não está no final): peticoes\102012005921_0000221201668837_200.txt
Pulando arquivo (procuração não está no final): peticoes\102012005921_0000221201668837_200.txt
Pulando arquivo (recibo não está no final): peticoes\102012030377_29409161947606360_207.txt
Pulando arquivo (instruções não está no final): peticoes\102012030377_29409161947606360_207.txt
Pulando arquivo (procuração não está no final): peticoes\102012030377_29409161947606360_207.txt
Pulando arquivo (recibo não está no final): peticoes\102013014262_29409161922930112_207.txt
Pulando arquivo (instruções não está no final): peticoes\102013014262_29409161922930112_207.txt
Pulando arquivo (procuração não está no final): peticoes\102013014262_29409161922930112_207.txt
Pulando arquivo (recibo não está no final): peticoes\102013019765_29409161942695631_207.txt
Pulando arquivo (instruções não está no final): peticoes\10

In [39]:
# isso aqui elimine quem não é recurso, não precisa executar

from datetime import datetime
import mysql.connector
import pandas as pd

conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

diretorio = "peticoes"
analisados = set()
data_alvo = datetime.strptime("01/04/2026", "%d/%m/%Y").date()
for nome_arquivo in os.listdir(diretorio):
    if nome_arquivo.lower().endswith(".txt"):
        arquivo_entrada = os.path.join(diretorio, nome_arquivo)
        timestamp = os.path.getmtime(arquivo_entrada)
        data_arquivo = datetime.fromtimestamp(timestamp).date()
        if data_arquivo != data_alvo:
            continue
        match = re.search(r'\\(\d+)_', arquivo_entrada)
        if match:
            numero = match.group(1)
            if numero in analisados:
                continue
            analisados.add(numero)
            comando = f"SELECT * FROM arquivados WHERE numero='{numero}' and despacho='12.2'"
            cursor.execute(comando)
            resultado = cursor.fetchall()
            if not resultado:
                print(numero)
                #print(arquivo_entrada)
                if nome_arquivo.startswith(numero):
                    caminho = os.path.join(diretorio, nome_arquivo)
                    #print(caminho)
                    os.remove(caminho)
                    print(f"Removido: {nome_arquivo}")
            else:
                df = pd.DataFrame(resultado)
                #print(f"recurso {numero}")



In [13]:
# atualize tabela anterioridades_ocorrencias
from datetime import datetime
diretorio = "peticoes"
data_alvo = datetime.strptime("11/04/2026", "%d/%m/%Y").date()
for nome_arquivo in os.listdir(diretorio):
    if nome_arquivo.lower().endswith(".txt"):
        arquivo_entrada = os.path.join(diretorio, nome_arquivo)
        arquivo_saida = arquivo_entrada
        timestamp = os.path.getmtime(arquivo_entrada)
        data_arquivo = datetime.fromtimestamp(timestamp).date()
        if data_arquivo != data_alvo:
            continue
        conteudo = ''
        with open(arquivo_entrada, 'r', encoding='utf-8') as f:
            conteudo = f.read()
        if "ilegalidade" in conteudo and "19/2023" in conteudo:
            with open(arquivo_entrada, 'r', encoding='utf-8') as f:
                linhas = f.readlines()
            numero = linhas[0].strip()
            linha_data = linhas[2].strip()
            data_str = linha_data.split(":")[1].strip()
            data = datetime.strptime(data_str, "%d/%m/%Y").date()
            linha_tipo = linhas[1].strip()
            tipo_peticao = linha_tipo.split("(")[1].split(")")[0]
            #print(f" {numero} {data} {tipo_peticao}")
            cmd = f"insert ignore into anterioridades_ocorrencias (id, numero, tipo, peticao, tipo_peticao, data_peticao) VALUES (null, '{numero}', 'ilegalidade', null, '{tipo_peticao}', '{data}');"; 
            print(cmd)
            #break


In [ ]:
# esta rotina esta identificando errado as anterioridades 
# 112012001157 D2 WO2009073497 ele junta indevidamente com o 11 da data
# esta gerando errado insert ignore into anterioridades (numero,codigo,doc,data) VALUES ('112012001157','D2','WO200907349711',null);

from urllib.parse import quote
import re

# SELECT * FROM `anterioridades_desc` WHERE descricao<>'' and razoes ='' and numero not in (select numero from anterioridades)
# https://jsonviewer.ai/



def corrigir_json_malformado(s):
    # Remove quebras de linha dentro de strings
    s = re.sub(r'(?<!\\)\n', r'\\n', s)
    s = re.sub(r'(?<!\\)\r', r'', s)
    return s

def extrair_patente3(descricao):
    match = re.search(r'([A-Z]{2})[\s\-]*(\d{4})/?(\d+)', descricao, re.IGNORECASE)
    
    if match:
        codigo = match.group(1).upper()
        parte1 = match.group(2)
        parte2 = match.group(3)
        return f"{codigo}{parte1}{parte2}"
    
    return None
    
import re

def extrair_patente2(texto):
    if not texto:
        return None

    # remove espaços entre números (ex: 0359 667 → 0359667)
    texto = re.sub(r'(\d)\s+(\d)', r'\1\2', texto)

    # padrão principal
    match = re.search(r'([A-Z]{2,3})[\s\-]*(\d{4})[\/\s]?(\d+)', texto, re.IGNORECASE)
    
    if match:
        return f"{match.group(1).upper()}{match.group(2)}{match.group(3)}"

    # fallback (casos tipo EP1158791)
    match = re.search(r'([A-Z]{2,3})(\d{6,})', texto, re.IGNORECASE)
    if match:
        return f"{match.group(1).upper()}{match.group(2)}"

    return None

def extrair_patente(texto):
    if not texto:
        return None

    # remove espaços entre números (ex: 0359 667 → 0359667)
    texto = re.sub(r'(\d)\s+(\d)', r'\1\2', texto)

    # Padrão para WO, US, EP com ano e número (ex: WO 2009 073497, US 2014 0139742)
    # Captura o código, ano e número, ignorando o que vem depois
    match = re.search(r'([A-Z]{2,3})\s+(\d{4})\s+(\d{4,7})', texto, re.IGNORECASE)
    if match:
        return f"{match.group(1).upper()}{match.group(2)}{match.group(3)}"

    # Padrão para US sem espaços (ex: US20130086607)
    match = re.search(r'([A-Z]{2,3})(\d{11,13})', texto, re.IGNORECASE)
    if match:
        return f"{match.group(1).upper()}{match.group(2)}"

    # Padrão para EP sem espaços (ex: EP1158791)
    match = re.search(r'([A-Z]{2,3})(\d{7,10})', texto, re.IGNORECASE)
    if match:
        return f"{match.group(1).upper()}{match.group(2)}"

    # Padrão para números com ano de 4 dígitos e número com 6 dígitos (ex: WO2009073497)
    match = re.search(r'([A-Z]{2,3})(\d{4})(\d{6})', texto, re.IGNORECASE)
    if match:
        return f"{match.group(1).upper()}{match.group(2)}{match.group(3)}"

    # Fallback genérico
    match = re.search(r'([A-Z]{2,3})(\d{6,})', texto, re.IGNORECASE)
    if match:
        return f"{match.group(1).upper()}{match.group(2)}"

    return None
    
def extrair_documentos(texto):
    # 1. Pega tudo após "Quadro 4"
    partes = re.split(r'Quadro\s*4', texto, flags=re.IGNORECASE)
    if len(partes) < 2:
        return []
    trecho = partes[1]
    
    # 2. Remove as linhas de cabeçalho
    linhas = trecho.split('\n')
    
    # Encontra onde começam os dados
    inicio = 0
    for i, linha in enumerate(linhas):
        if re.search(r'\bD\d+\b', linha):
            inicio = i
            break
    
    # Processa as linhas a partir do início
    documentos = []
    i = inicio
    
    while i < len(linhas):
        linha_atual = linhas[i]
        
        # Verifica se a linha atual contém um código D
        match_codigo = re.match(r'^\s*(D\d+)\s+(.*)', linha_atual)
        
        if match_codigo:
            codigo = match_codigo.group(1)
            descricao = match_codigo.group(2).strip()
            
            # Avança para a próxima linha
            i += 1
            
            # Continua adicionando linhas enquanto não encontrar um novo código D
            while i < len(linhas):
                proxima_linha = linhas[i]
                
                # Se a próxima linha começa com D (novo documento), para
                if re.match(r'^\s*D\d+', proxima_linha):
                    break
                
                # Adiciona o conteúdo da linha à descrição
                if proxima_linha.strip():
                    descricao += " " + proxima_linha.strip()
                
                i += 1
            
            # Limpa a descrição: remove múltiplos espaços
            descricao = " ".join(descricao.split())
            
            # ANTES de extrair a patente, remove os dígitos soltos no final
            # Ex: "WO 2009 073497 11" -> "WO 2009 073497"
            descricao = re.sub(r'\s+\d+$', '', descricao)
            
            # Agora extrai a patente
            descricao = extrair_patente(descricao)
            
            documentos.append({
                "codigo": codigo,
                "descricao": descricao
            })
        else:
            i += 1
    
    return documentos


query = '"mysql_query":" * FROM anterioridades_desc where descricao<>\'\' and razoes=\'\' "'
query = '"mysql_query":" * FROM anterioridades_desc where descricao<>\'\' and razoes=\'\' and numero=\'112012001157\' "'
query_encoded = quote(query, safe='')
url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query_encoded}"
print(url)

json_data = conectar_siscap(url)

if isinstance(json_data, str):
    json_data = corrigir_json_malformado(json_data)
    json_data = json.loads(json_data)
    #print(json_data)

if not json_data:
    raise ValueError("Resposta vazia da API")

for i in range(0, len(json_data["patents"])):
    numero = json_data["patents"][i]['numero'] # para ler a lista de numeros especificos
    #print(numero)
    descricao = json_data["patents"][i]['descricao']
    print(descricao)
    documentos = extrair_documentos(descricao)
    if documentos != []:
        validos = [doc for doc in documentos if doc['descricao']]
        #print(documentos)
        for x in documentos:
            codigo = x['codigo']
            doc = x['descricao']
            if doc is not None:
                doc = doc.replace('BR','')
                doc = doc[:25].strip()
                #doc = extrair_patente(doc)
                cmd = f"insert ignore into anterioridades (numero,codigo,doc,data) VALUES ('{numero}','{codigo}','{doc}',null);"
                print(cmd)
                #break
#102012028228 indeferimento técnico



In [ ]:
#identifique registros que estao com conclusao vazia

import mysql.connector
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

comando = f"SELECT * FROM `carga` WHERE numero in (select numero from arquivados where despacho='12.2' and anulado=0) and numero in (select numero from anterioridades_desc where conclusao='')"
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)
lista = df.values.tolist()
if df.shape[1] > 1:
    lista = df.iloc[:, 0].tolist()
else:
    lista = []
lista.insert(0, 'numero')

#lista=['numero','PI0922730']
print(lista)

In [ ]:
# atualiza campo conclusao de anterioridades_desc

import os, re
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts.prompt import PromptTemplate

def limpar_caracteres_especiais(texto):
    # Remove caracteres não imprimíveis
    texto = re.sub(r'[^\x20-\x7EÀ-ÿ]', '', texto)
    return texto
 
def remove_page_and_pid(text: str) -> str:
    # remove ocorrência "Página <n>" possivelmente seguida por código PI...
    text = re.sub(r'(?i)\bPágina\s*\d+(?:\s*(?:de|/)\s*\d+)?', '', text)
    # remove ocorrência "Página <n> PI..." (se sobrar algum resíduo)
    text = re.sub(r'(?i)\bPágina\s*\d+(?:\s+PI\d+(?:-\d+)?)?', '', text)
    # remove códigos PI isolados como "PI1009860-7" ou "PI 1009860-7"
    text = re.sub(r'\bPI\s*\d+(?:-\d+)?\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\bBR\s*\d+(?:-\d+)?\b', '', text, flags=re.IGNORECASE)
    # remove sequências de espaços em branco extras
    text = re.sub(r'\s{2,}', ' ', text)
    # remove espaços antes de pontuação
    text = re.sub(r'\s+([,.;:!?])', r'\1', text)
    # remover traços/lists soltos do tipo " - " ou " — " que ficaram isolados
    text = re.sub(r'\s*[-–—]\s*(?=[^\w-])', ' ', text)
    # remover traços isolados no começo ou fim de linhas/frases
    text = re.sub(r'^[\s\-–—]+', '', text)
    text = re.sub(r'[\s\-–—]+$', '', text)
    # remover espaços duplos novamente e aparar
    text = re.sub(r'\s{2,}', ' ', text).strip()
    text = text.replace("..",".")
    text = text.replace(". .",".")
    text = text.replace("Art .8","Art 8")
    return text

def format_as_single_paragraph(text):
    # Remove quebras de linha e espaços extras
    formatted_text = ' '.join(line.strip() for line in text.splitlines() if line.strip())
    return formatted_text
    
with open("descricao.sql", "a", encoding="utf-8") as f:
    for i in range(1, len(lista)): # começa em 1 porque pula numero
        #numero = data["patents"][i]["numero"]
        numero = lista[i] # para ler a lista de numeros especificos
    
        #query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM pedido where decisao='indeferimento' and numero='{numero}'" + '"'
        query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM pedido where (decisao='indeferimento' or decisao='ciencia de parecer') and numero='{numero}'" + ' order by rpi desc"'
        url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
        print(url)
        try:
            json_data = conectar_siscap(url,return_json=True)
            data1 = json.loads(json_data)
            codigo = data1["patents"][0]["codigo"]
            divisao = data1["patents"][0]["divisao"]    
            
            caminho_do_arquivo = f"pareceres/{numero}{codigo}.txt"
            if os.path.exists(caminho_do_arquivo):
                with open(caminho_do_arquivo, 'r', encoding='utf-8') as arquivo:
                    texto_relatorio = arquivo.read()
            else:
                print(f"Arquivo não encontrado: {caminho_do_arquivo}")
                texto_relatorio = None  # ou "" dependendo do seu uso
            
            #match = re.search(r"(CONCLUS[aã]O\s*[\r\n]+.*?)(?:Rio de Janeiro|$)", texto_relatorio, flags=re.S | re.I)
            match = re.search(
                r"^\s*[-–—]?\s*CONCLUS[aã]O\s*:?\s*$\s*(.*?)(?:^\s*Rio de Janeiro|\Z)",
                texto_relatorio,
                flags=re.S | re.I | re.M
            )
    
            if match:
                conclusao = re.sub(r"\s+", " ", match.group(1)).strip()
                print("Conclusão extraída:\n")
                conclusao = conclusao.replace("Conclusão", "")
                conclusao = conclusao.replace("CONCLUSÃO", "")
                conclusao = conclusao.replace("Assim sendo,", "")
                conclusao = conclusao.replace("","")
                conclusao = conclusao.replace("","")
                conclusao = conclusao.replace("","")
                conclusao = conclusao.replace("O depositante deve se manifestar quanto ao contido neste parecer em até 90 (noventa) dias, a partir da data de publicação na RPI, de acordo com o Art. 36 da LPI","")
                conclusao = conclusao.replace("Publique-se a ciência de parecer (7.1).","")
                conclusao = conclusao.split("De acordo com o Art. 212")[0].strip()
                conclusao = conclusao[0].upper() + conclusao[1:]
                conclusao = conclusao.replace("'","")
                conclusao = remove_page_and_pid(conclusao)
                conclusao = conclusao.replace("Código:5975ce1e23737deeb22c88e902a79153versão1.3 19/04/12","")
                conclusao = conclusao.replace('Código:5975ce1e23737deeb22c88e902a79153versão1.1 27/01/12','')
                conclusao = limpar_caracteres_especiais(conclusao)
                sql = f"update anterioridades_desc set conclusao='{conclusao}' where numero='{numero}';"
                print(sql)
                f.write(sql + "\n")
            elif "Assim sendo, de acordo com o Art. 37" in texto_relatorio:
                pos = texto_relatorio.find("Assim sendo, de acordo com o Art. 37")
                conclusao = texto_relatorio[pos:]
                conclusao = conclusao.strip()
                print("Conclusão extraída:\n")
                conclusao = conclusao.replace("Conclusão", "")
                conclusao = conclusao.replace("CONCLUSÃO", "")
                conclusao = conclusao.replace("Assim sendo,", "")
                conclusao = conclusao.replace("","")
                conclusao = conclusao.replace("","")
                conclusao = conclusao.replace("","")
                conclusao = conclusao.replace("O depositante deve se manifestar quanto ao contido neste parecer em até 90 (noventa) dias, a partir da data de publicação na RPI, de acordo com o Art. 36 da LPI","")
                conclusao = conclusao.replace("Publique-se a ciência de parecer (7.1).","")
                conclusao = conclusao.split("De acordo com o Art. 212")[0].strip()
                conclusao = conclusao[0].upper() + conclusao[1:]
                conclusao = conclusao.replace("'","")
                conclusao = remove_page_and_pid(conclusao)
                conclusao = conclusao.replace("Código:5975ce1e23737deeb22c88e902a79153versão1.3 19/04/12","");
                conclusao = conclusao.replace('Código:5975ce1e23737deeb22c88e902a79153versão1.1 27/01/12','')
                conclusao = limpar_caracteres_especiais(conclusao)
                sql = f"update anterioridades_desc set conclusao='{conclusao}' where numero='{numero}';"
                print(sql)
                f.write(sql + "\n")
            else:
                print(f"Trecho não encontrado {numero}.")
        
        except Exception as e:
            print(f"Não achei parecer de indeferimento {numero} {e}")

In [ ]:
# certifique-se que todos os registros da carga tem conclusao preenchida
# SELECT * FROM `carga` WHERE numero in (select numero from arquivados where despacho='12.2' and anulado=0) and numero in (select numero from anterioridades_desc where conclusao='' and modelo='gpt-5-nano')
# ajuste manualmente os registros ainda sem conclusao
# procure Find in Files *.txt cpf cnpj hyaip.com dbba.com vcpi.com www.daniel gruenbaim.com 

In [48]:

# atualização da tabela anterioridades com os documentos do estado da tecnica

import mysql.connector
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

comando = f"SELECT * FROM `carga` WHERE numero in (select numero from arquivados where despacho='12.2' and anulado=0) and numero not in (select numero from anterioridades)"
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)
lista = df.values.tolist()
if df.shape[1] > 1:
    lista = df.iloc[:, 0].tolist()
else:
    lista = []
lista.insert(0, 'numero')
print(lista)

['numero', 'PI0921237', 'PI0920800', 'PI0823184', '112013019699', '112013017629', '112013015675', '112012015325', '102023003227', '102019023319', '112014033067', '112014030713', '102016029482', '112014027283', '112014026149', '112014020793', '102015028831', '102015003367', '102014031535', '102014031388', '112014001086', '102013009314', '122020015894', '122020015741', '122020014755', '122020014740', '122020013811', '122020010712', '122020008510', '122019026829', '112017013553', 'PI1103928', 'PI1004183', '122019001140', '112017005975', '112017001373', '112017000259', '112016026350', '112016024777', '112016023062', '122017012058', '112016002195', '112015032570', 'PI0306160', '202019023953', '112015006953', '202013002255', '112020011171', '202012023481', '122022017195', '122022017178', '122021020395', '122021015578', '122021014646', '122021007541', '122020021367', '122020017521', 'PI0818286', '102012012377', '102012004079', '112013009746', '112013004702', '112012026730', '112015004775', '1

In [ ]:
#atualiza tabela anterioridades

# ******************************************
# gera lista de docs para cada um dos pedidos na carga da tabela anterioridades. Não usa LLM
# rodar rotina abaixo com VPN ligada certifique-se que siscap.inpi.gov.br funcionando

import re
from datetime import datetime
import json
import requests  # Supondo que conectar_siscap use requests

def converter_data(data):
    if not data:
        return None
    # Remove qualquer coisa que não seja número ou /
    data = re.sub(r'[^0-9/]', '', data)
    formatos = ['%d/%m/%y', '%d/%m/%Y']
    for fmt in formatos:
        try:
            return datetime.strptime(data, fmt).strftime('%Y-%m-%d')
        except ValueError:
            pass  # tenta o próximo formato
    return None

def montar_url_parecer(numero):
    try:
        query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM pedido where (decisao='indeferimento' or decisao='ciencia de parecer') and numero='{numero}'" + ' order by rpi desc"'
        url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
        print(url)
        json_data = conectar_siscap(url,return_json=True)
        if json_data is not None:
            data1 = json.loads(json_data)
            codigo = data1["patents"][0]["codigo"]
            divisao = data1["patents"][0]["divisao"]  
            url = f"pareceres/{numero}{codigo}.txt"
            return url
        else:
            return None
    except Exception as e:
        print(f"Não achei parecer de indeferimento {numero} {e}")
            
#    query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM pedido where (decisao='indeferimento' or decisao='ciencia de parecer') and numero='{numero}' order by rpi desc" + '"'
#    url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
#    json_data = conectar_siscap(url,return_json=True)
#    if json_data is not None:
#        data = json.loads(json_data)
#        if not json_data or "patents" not in json_data or len(data["patents"]) == 0:
#            return None
#        else:
#            data = json.loads(json_data)
#            codigo = data["patents"][0]["codigo"]
#            divisao = data["patents"][0]["divisao"]
#            # print(f"Código: {codigo}")
#            # print(f"Divisão: {divisao}")
#            url = f"https://siscap.inpi.gov.br/adm/pareceres/{divisao}/{numero}{codigo}.txt"
#            url = f"pareceres/{numero}{codigo}.txt"
#            return url
#    else:
#        return None

saida = ''
url = ''
total = 0
query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM carga WHERE divisao='direp'" + '"'
url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
json_data = conectar_siscap(url,return_json=True)
data = json.loads(json_data)
numbers = [patent['numero'] for patent in data['patents']]
#numbers = ['102019009508']
#lista = ['202014005715']
numbers = lista

with open("descricao.sql", "a", encoding="utf-8") as f:
    for numero in numbers:
        total = total + 1
        if (total>1000):
            break
        url = montar_url_parecer(numero)
        if url is not None:
            print(numero)
            print(url)
            
            #texto_relatorio = conectar_siscap(url,return_json=False)
            caminho_do_arquivo = url
            if os.path.exists(caminho_do_arquivo):
                with open(caminho_do_arquivo, 'r', encoding='utf-8') as arquivo:
                    texto_relatorio = arquivo.read()
            else:
                print(f"Arquivo não encontrado: {caminho_do_arquivo}")
                texto_relatorio = None  # ou "" dependendo do seu uso

            #print(texto_relatorio)
            if texto_relatorio is not None:
                #caminho_do_arquivo='document.txt'
                #with open(caminho_do_arquivo, 'w', encoding='utf-8') as arquivo:
                #    arquivo.write(texto_relatorio)
    
                pattern = r"(D\d+)\s+((?:[A-Z]{2,3})\s*\d{4,10}(?:-\d[A-Z]?)?)\s+.*?(\d{2}[\/\.]\d{2}[\/\.]\d{4})"
                pattern = r"(D\d+)\s+([A-Z]{2,3}\s*\d[\d\.,]*?)\s+(\d{2}[\/\.]\d{2}[\/\.]\d{4})"
                pattern = r"(D\d+)\s+([A-Z]{2,3}\d+\s*[A-Z]?\d?)\s+(\d{2}[\/\.]\d{2}[\/\.]\d{4})"
                pattern = r"(D\d+)\s+([A-Z]{2,3}\s*\d+(?:-\d+)?)\s+(\d{2}[\/\.]\d{2}[\/\.]\d{4})"
                pattern =   r"""
                            (D\d+)                                      # Código D1, D2...
                            \s+
                            (
                                [A-Z]{2,3}                               # País (PI, BR, US, WO, EP...)
                                \s*
                                \d{4,}                                   # Número principal (mín 4 dígitos)
                                (?:[\/\-]\d+)?                           # Parte opcional tipo 2019/123456 ou -2
                                (?:\s*[A-Z]\d)?                          # Sufixo opcional tipo A1, B1
                            )
                            [\s\S]*?
                            (\d{2}[\/\.]\d{2}[\/\.]\d{4})                 # Data
                            """
    
                #matches = re.findall(pattern, texto_relatorio)
                matches = re.findall(pattern, texto_relatorio, re.VERBOSE | re.IGNORECASE)
                for match in matches:
                    #codigo, documento, tipo, data = match
                    codigo = match[0]
                    documento = match[1]
                    doc = " ".join(documento.split())
                    doc = doc.replace(" ", "")
                    doc = doc.replace("/", "")
                    doc = doc.replace(".", "")
                    doc = doc.replace(",", "")
                    doc = doc.replace(";", "")
                    doc = doc.split("-")[0]
                    data = match[-1]
                    data = converter_data(data)
                    #data = datetime.strptime(data, '%d/%m/%Y') # 11/07/2013
                    #data = data.strftime('%Y-%m-%d')
                    ##print(f"{codigo}: {doc}, Data de Publicação = {data}")
                    sql = f"INSERT IGNORE INTO anterioridades (numero, codigo, doc, data) VALUES ('{numero}','{codigo}','{doc}','{data}');"
                    print(sql)
                    saida = saida + f"INSERT IGNORE INTO anterioridades (numero, codigo, doc, data) VALUES ('{numero}','{codigo}','{doc}','{data}');\n"
                    f.write(sql + "\n")

print(saida)
# teste regex https://regex101.com/

In [ ]:
# atualize anterioridades_desc com os respoectivos registrso para receber gpt-5-nano, qwen/qwen3-32b, 'gpt-oss-20b e gemini-2.5-flash
# https://cientistaspatentes.com.br/central/control.php?action=190&op=15

In [50]:
# rode https://cientistaspatentes.com.br/central/control.php?action=190&op=13
# certifique-se de não haver nenhum erro e que a query seguinte seja vazia, ou seja, todos registros tem campo razoes preenchido
# SELECT * FROM `carga` WHERE numero in (select numero from arquivados where despacho='12.2' and anulado=0) and numero in (select numero from anterioridades_desc where razoes='')
# rode https://cientistaspatentes.com.br/central/control.php?action=190&op=12
# certifique-se de que toos os registros tem historico pedido
# SELECT * FROM `carga` WHERE numero in (select numero from arquivados where despacho='12.2' and anulado=0) and numero in (select numero from anterioridades_desc where historico_pedido='')

# Atualize campo incoerencia da tabela anterioridades_desc

import mysql.connector
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

comando = f"SELECT * FROM `carga` WHERE numero in (select numero from arquivados where despacho='12.2' and anulado=0) and numero in (select numero from anterioridades_desc where incoerencia='' and modelo='qwen/qwen3-32b')"
comando = f"SELECT * FROM `carga` WHERE examinador in ('abrantes','milavsl','scaplan','srosa','cidade','eloliveira','apedrosa','rosanab','jsoares','alciclea','szandona','douglasm','cujikawa') and numero in (select numero from arquivados where despacho='12.2' and anulado=0) and numero in (select numero from anterioridades_desc where incoerencia='' and modelo='qwen/qwen3-32b')"
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)
lista = df.values.tolist()
if df.shape[1] > 1:
    lista = df.iloc[:, 0].tolist()
else:
    lista = []
lista.insert(0, 'numero')
print(lista)

['numero', '112018008824', '112014008059', '202014026469']


In [51]:
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import re
def clean_answer(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    return text.strip()
    
load_dotenv(dotenv_path='.env', override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")
# lista = ['202016018807']

In [52]:
numero = '122021008506'
comando = f"SELECT conclusao FROM anterioridades_desc WHERE numero='{numero}'"
cursor.execute(comando)
resultado = cursor.fetchall()
print(resultado[0][0])

De acordo com o Art. 37, indefiro o presente pedido, uma vez que: não atende ao requisito de atividade inventiva (Art.8 combinado com Art. 13 da LPI) as reivindicações estão indefinidas e/ou não estão fundamentadas no relatório descritivo (Art. 25 da LPI)


In [53]:
print(len(lista))

4


In [ ]:
# atualiza campo incoerencia com qwen/qwen3-32b

import mysql.connector
import time
from datetime import date

hoje = date.today()

json_data = {"patents": [{"numero": item} for item in lista]}
json_data = json.dumps(json_data, indent=4, ensure_ascii=False)
data = json.loads(json_data)

with open("descricao.sql", "a", encoding="utf-8") as f:
    for patent in data.get("patents", []):
        numero = patent.get("numero")
        if numero != 'NUMERO':
            query_pedido = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM pedido where decisao='indeferimento' and numero='{numero}'" + '"'
            url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query_pedido}"
            print(url)
            try:
                json_data = conectar_siscap(url,return_json=True)
            except:
                print(f"conexão inválida, pulando (indeferimento {numero} não encontrado)...")
                continue

            #print(json_data)
            if not json_data:
                print("json_data vazio, pulando...")
                continue
            try:
                data_pedido = json.loads(json_data)
            except json.JSONDecodeError:
                print("JSON inválido, pulando...")
                continue
    
            if not data_pedido["patents"]:
                print(f"Sem registros para {numero}, pulando...")
                continue
        
            #data_pedido = json.loads(json_data)
            codigo = data_pedido["patents"][0]["codigo"]
            divisao = data_pedido["patents"][0]["divisao"]
        
            #url = f"https://siscap.inpi.gov.br/adm/pareceres/{divisao}/{numero}{codigo}.txt"
            #print(url)
            #texto_relatorio = conectar_siscap(url, return_json=False)
            #print(texto_relatorio)

            url = f"pareceres/{numero}{codigo}.txt"
            caminho_do_arquivo = url
            if os.path.exists(caminho_do_arquivo):
                with open(caminho_do_arquivo, 'r', encoding='utf-8') as arquivo:
                    texto_relatorio = arquivo.read()
            else:
                print(f"Arquivo não encontrado: {caminho_do_arquivo}")
                texto_relatorio = None  # ou "" dependendo do seu uso

            comando = f"SELECT conclusao FROM anterioridades_desc WHERE numero='{numero}'"
            cursor.execute(comando)
            resultado = cursor.fetchall()
            conclusao = resultado[0][0]

            query = f"""Neste parecer verifique se existência de incoerência interna entre as conclusões: [{conclusao}] e o restante do parecer. 
            Ignore os X nos quadros 2 e 3 e considere apenas a discussão que se segue nestes quadros (se houver).
            Quando apenas uma ou mais reivindicação não tem novidade ou atividade inventiva, é justificável indeferir o pedido por falta de novidade 
            ou atividade inventiva, respectivamente. Escreva a saída numa forma corrida, sem bullets, parágrafos, travessões, nem pula linha.
            Apresente uma resposta curta e objetiva. Relatório: {texto_relatorio}"""

            MAX_CHARS = 12000  # ajuste conforme necessário
            if len(query) > MAX_CHARS:
                query = query[:MAX_CHARS]
                
            llm = ChatGroq(model="qwen/qwen3-32b")
            messages=[{"role":"user", "content": query}]
            response = llm.invoke(messages)
            resumo = clean_answer(response.content)
            time.sleep(3) 

            #url = "https://api.openai.com/v1/chat/completions"
            #data_json = {
            #    "model": "gpt-5-mini",  # Use o modelo desejado, como 'gpt-5.2' ou 'gpt-4o-mini' ou 'gpt-5-mini'
            #    "messages": [
            #        {"role": "user", "content": query}
            #    ]
            #}
            #headers = {
            #    "Authorization": f"Bearer {openai_api_key}",
            #    "Content-Type": "application/json"
            #}
            #response = requests.post(url, headers=headers, json=data_json, verify=False)
            
            #if response.status_code != 200:
            #    print(f"Não consegui conexão {numero} (HTTP {response.status_code})")
            #    continue
            
            #try:
            #    resposta = response.json()
            #    resumo = resposta["choices"][0]["message"]["content"]
            #except (ValueError, KeyError, IndexError, TypeError):
            #    print(f"Resposta inválida da API para {numero}")
            #    continue
                
            texto_corrigido = " ".join(resumo.split())
            texto_corrigido = texto_corrigido.replace("'", "")
            texto_corrigido = texto_corrigido.replace("‑","-")
            sql_resumo = f"UPDATE anterioridades_desc SET incoerencia='{texto_corrigido}', data='{hoje}' WHERE numero='{numero}' and modelo='qwen/qwen3-32b';"
            print(sql_resumo)
            f.write(sql_resumo + "\n")
            #break

In [55]:
# faz a atualização do campo incoerencia com gpt apenas nos pedidos de abrantes com gpt-5-nano

import mysql.connector
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

comando = f"SELECT * FROM `carga` WHERE examinador='abrantes' and numero in (select numero from arquivados where despacho='12.2' and anulado=0) and numero in (select numero from anterioridades_desc where incoerencia='' and modelo='gpt-5-nano')"
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)
lista = df.values.tolist()
if df.shape[1] > 1:
    lista = df.iloc[:, 0].tolist()
else:
    lista = []
lista.insert(0, 'numero')
print(lista)

['numero']


In [38]:
print(len(lista))

13


In [ ]:
# atualiza campo incoerencia com gpt-5-nano os registros de abrantes apenas

import mysql.connector
import time
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from datetime import date

hoje = date.today()

load_dotenv(dotenv_path='.env', override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
llm = ChatOpenAI(model="gpt-5-nano", openai_api_key=openai_api_key,temperature=1)

json_data = {"patents": [{"numero": item} for item in lista]}
json_data = json.dumps(json_data, indent=4, ensure_ascii=False)
data = json.loads(json_data)

with open("descricao.sql", "a", encoding="utf-8") as f:
    for patent in data.get("patents", []):
        numero = patent.get("numero")
        if numero != 'NUMERO':
            query_pedido = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM pedido where decisao='indeferimento' and numero='{numero}'" + '"'
            url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query_pedido}"
            print(url)
            try:
                json_data = conectar_siscap(url,return_json=True)
            except:
                print(f"conexão inválida, pulando (indeferimento {numero} não encontrado)...")
                continue

            #print(json_data)
            if not json_data:
                print("json_data vazio, pulando...")
                continue
            try:
                data_pedido = json.loads(json_data)
            except json.JSONDecodeError:
                print("JSON inválido, pulando...")
                continue
    
            if not data_pedido["patents"]:
                print(f"Sem registros para {numero}, pulando...")
                continue
        
            #data_pedido = json.loads(json_data)
            codigo = data_pedido["patents"][0]["codigo"]
            divisao = data_pedido["patents"][0]["divisao"]
        
            #url = f"https://siscap.inpi.gov.br/adm/pareceres/{divisao}/{numero}{codigo}.txt"
            #print(url)
            #texto_relatorio = conectar_siscap(url, return_json=False)
            #print(texto_relatorio)

            url = f"pareceres/{numero}{codigo}.txt"
            caminho_do_arquivo = url
            if os.path.exists(caminho_do_arquivo):
                with open(caminho_do_arquivo, 'r', encoding='utf-8') as arquivo:
                    texto_relatorio = arquivo.read()
            else:
                print(f"Arquivo não encontrado: {caminho_do_arquivo}")
                texto_relatorio = None  # ou "" dependendo do seu uso

            comando = f"SELECT conclusao FROM anterioridades_desc WHERE numero='{numero}'"
            cursor.execute(comando)
            resultado = cursor.fetchall()
            conclusao = resultado[0][0]

            query = f"""Neste parecer verifique se existência de incoerência interna entre as conclusões: [{conclusao}] e o restante do parecer. 
            Ignore os X nos quadros 2 e 3 e considere apenas a discussão que se segue nestes quadros (se houver).
            Quando apenas uma ou mais reivindicação não tem novidade ou atividade inventiva, é justificável indeferir o pedido por falta de novidade 
            ou atividade inventiva, respectivamente. Escreva a saída numa forma corrida, sem bullets, parágrafos, travessões, nem pula linha.
            Apresente uma resposta curta e objetiva. Relatório: {texto_relatorio}"""
              
            messages=[{"role":"user", "content": query}]
            response = llm.invoke(messages)
            resumo = response.content

            #url = "https://api.openai.com/v1/chat/completions"
            #data_json = {
            #    "model": "gpt-5-mini",  # Use o modelo desejado, como 'gpt-5.2' ou 'gpt-4o-mini' ou 'gpt-5-mini'
            #    "messages": [
            #        {"role": "user", "content": query}
            #    ]
            #}
            #headers = {
            #    "Authorization": f"Bearer {openai_api_key}",
            #    "Content-Type": "application/json"
            #}
            #response = requests.post(url, headers=headers, json=data_json, verify=False)
            
            #if response.status_code != 200:
            #    print(f"Não consegui conexão {numero} (HTTP {response.status_code})")
            #    continue
            
            #try:
            #    resposta = response.json()
            #    resumo = resposta["choices"][0]["message"]["content"]
            #except (ValueError, KeyError, IndexError, TypeError):
            #    print(f"Resposta inválida da API para {numero}")
            #    continue
                
            texto_corrigido = " ".join(resumo.split())
            texto_corrigido = texto_corrigido.replace("'", "")
            texto_corrigido = texto_corrigido.replace("‑","-")
            sql_resumo = f"UPDATE anterioridades_desc SET incoerencia='{texto_corrigido}', data='{hoje}' WHERE numero='{numero}' and modelo='gpt-5-nano';"
            print(sql_resumo)
            f.write(sql_resumo + "\n")
            #break

In [22]:
# atualiza campo incoerencia com gpt-oss-20b
import mysql.connector
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

comando = f"SELECT * FROM `carga` WHERE numero in (select numero from arquivados where despacho='12.2' and anulado=0) and numero in (select numero from anterioridades_desc where incoerencia='' and modelo='gpt-oss-20b')"
comando = f"SELECT * FROM `carga` WHERE examinador in ('abrantes','milavsl','scaplan','srosa','cidade','eloliveira','apedrosa','rosanab','jsoares','alciclea','szandona','douglasm','cujikawa') and numero in (select numero from arquivados where despacho='12.2' and anulado=0) and numero in (select numero from anterioridades_desc where incoerencia='' and modelo='gpt-oss-20b')"
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)
lista = df.values.tolist()
if df.shape[1] > 1:
    lista = df.iloc[:, 0].tolist()
else:
    lista = []
lista.insert(0, 'numero')
print(lista)

['numero', 'MU9100240', 'PI0906951', '102023009444', '112015005351', '112015005153', '112015000808', '112014030449', '102016027868', '102015029049', '102015007934', '112014008059', '112014000676', '102014002048', '102013025651', '102013014262', '102012030377', '122020001985', '122019026850', '122019016068', '112016003968', '112015030352', '112015027282', 'PI1013698', '202020026256', '202020007390', '112021023943', '112021011264', '112015011550', '202016018807', '202013019250', '122022020636', '122022017789', '122020017517', '112018005056', '112013019993', '112012029897', '112012026999', '112012025948', '112012022998', '112012007444', '102020001796', '112015003527', '112015002586', '112014032938', '112014031962', '102016015976', '112014019190', '102015031651', '102015031507', '102015030787', '112014011387', '112014008804', '102014022484', '102014021906', '112014007484', '112014006871', '112014004260', '112014000630', '102014006371', '122020009103', '122019024740', '112016028749', '11201

In [23]:
print(len(lista))

75


In [ ]:
# atualiza campo incoerencia com gpt-oss-20b

import mysql.connector
import time
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
from datetime import date

hoje = date.today()

load_dotenv(dotenv_path='.env', override=True)
groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(model="openai/gpt-oss-20b",groq_api_key=groq_api_key,temperature=0)

json_data = {"patents": [{"numero": item} for item in lista]}
json_data = json.dumps(json_data, indent=4, ensure_ascii=False)
data = json.loads(json_data)

with open("descricao.sql", "a", encoding="utf-8") as f:
    for patent in data.get("patents", []):
        numero = patent.get("numero")
        if numero != 'NUMERO':
            query_pedido = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM pedido where decisao='indeferimento' and numero='{numero}'" + '"'
            url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query_pedido}"
            print(url)
            try:
                json_data = conectar_siscap(url,return_json=True)
            except:
                print(f"conexão inválida, pulando (indeferimento {numero} não encontrado)...")
                continue

            #print(json_data)
            if not json_data:
                print("json_data vazio, pulando...")
                continue
            try:
                data_pedido = json.loads(json_data)
            except json.JSONDecodeError:
                print("JSON inválido, pulando...")
                continue
    
            if not data_pedido["patents"]:
                print(f"Sem registros para {numero}, pulando...")
                continue
        
            #data_pedido = json.loads(json_data)
            codigo = data_pedido["patents"][0]["codigo"]
            divisao = data_pedido["patents"][0]["divisao"]
        
            #url = f"https://siscap.inpi.gov.br/adm/pareceres/{divisao}/{numero}{codigo}.txt"
            #print(url)
            #texto_relatorio = conectar_siscap(url, return_json=False)
            #print(texto_relatorio)

            url = f"pareceres/{numero}{codigo}.txt"
            caminho_do_arquivo = url
            if os.path.exists(caminho_do_arquivo):
                with open(caminho_do_arquivo, 'r', encoding='utf-8') as arquivo:
                    texto_relatorio = arquivo.read()
            else:
                print(f"Arquivo não encontrado: {caminho_do_arquivo}")
                texto_relatorio = None  # ou "" dependendo do seu uso

            comando = f"SELECT conclusao FROM anterioridades_desc WHERE numero='{numero}'"
            cursor.execute(comando)
            resultado = cursor.fetchall()
            conclusao = resultado[0][0]

            query = f"""Neste parecer verifique se existência de incoerência interna entre as conclusões: [{conclusao}] e o restante do parecer. 
            Ignore os X nos quadros 2 e 3 e considere apenas a discussão que se segue nestes quadros (se houver).
            Quando apenas uma ou mais reivindicação não tem novidade ou atividade inventiva, é justificável indeferir o pedido por falta de novidade 
            ou atividade inventiva, respectivamente. Escreva a saída numa forma corrida, sem bullets, parágrafos, travessões, nem pula linha.
            Apresente uma resposta curta e objetiva. Relatório: {texto_relatorio}"""

            MAX_CHARS = 12000  # ajuste limite de 8000 tokens ≈ 24.000 a 32.000 caracteres
            if len(query) > MAX_CHARS:
                query = query[:MAX_CHARS]
                
            messages=[{"role":"user", "content": query}]
            response = llm.invoke(messages)
            resumo = response.content
            time.sleep(3) 

            #url = "https://api.openai.com/v1/chat/completions"
            #data_json = {
            #    "model": "gpt-5-mini",  # Use o modelo desejado, como 'gpt-5.2' ou 'gpt-4o-mini' ou 'gpt-5-mini'
            #    "messages": [
            #        {"role": "user", "content": query}
            #    ]
            #}
            #headers = {
            #    "Authorization": f"Bearer {openai_api_key}",
            #    "Content-Type": "application/json"
            #}
            #response = requests.post(url, headers=headers, json=data_json, verify=False)
            
            #if response.status_code != 200:
            #    print(f"Não consegui conexão {numero} (HTTP {response.status_code})")
            #    continue
            
            #try:
            #    resposta = response.json()
            #    resumo = resposta["choices"][0]["message"]["content"]
            #except (ValueError, KeyError, IndexError, TypeError):
            #    print(f"Resposta inválida da API para {numero}")
            #    continue
                
            texto_corrigido = " ".join(resumo.split())
            texto_corrigido = texto_corrigido.replace("'", "")
            texto_corrigido = texto_corrigido.replace("‑","-")
            sql_resumo = f"UPDATE anterioridades_desc SET incoerencia='{texto_corrigido}', data='{hoje}' WHERE numero='{numero}' and modelo='gpt-oss-20b';"
            print(sql_resumo)
            f.write(sql_resumo + "\n")
            #break

In [ ]:
# atualiza campo incoerencia com gpt-oss-20b
import mysql.connector
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

comando = f"SELECT * FROM `carga` WHERE numero in (select numero from arquivados where despacho='12.2' and anulado=0) and numero in (select numero from anterioridades_desc where incoerencia='' and modelo='gemini-2.5-flash')"
#comando = f"SELECT * FROM `carga` WHERE numero in (select numero from arquivados where despacho='12.2' and anulado=0)"
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)
lista = df.values.tolist()
if df.shape[1] > 1:
    lista = df.iloc[:, 0].tolist()
else:
    lista = []
lista.insert(0, 'numero')
print(lista)

In [54]:
print(len(lista))

667


In [12]:
# faz a atualização do campo incoerencia apenas nos pedidos de abrantes com gemini-2.5-flash

import mysql.connector
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

comando = f"SELECT * FROM `carga` WHERE examinador='abrantes' and numero in (select numero from arquivados where despacho='12.2' and anulado=0) and numero in (select numero from anterioridades_desc where incoerencia='' and modelo='gemini-2.5-flash')"
comando = f"SELECT * FROM `carga` WHERE examinador in ('abrantes','milavsl','scaplan','srosa','cidade','eloliveira','apedrosa','rosanab','jsoares','alciclea','szandona','douglasm','cujikawa') and numero in (select numero from arquivados where despacho='12.2' and anulado=0) and numero in (select numero from anterioridades_desc where incoerencia='' and modelo='gemini-2.5-flash')"
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)
lista = df.values.tolist()
if df.shape[1] > 1:
    lista = df.iloc[:, 0].tolist()
else:
    lista = []
lista.insert(0, 'numero')
print(lista)

['numero', '122019001140', '112015032570', '112022008919', '112015023151', '112015021619', '112015020849', '112015008759', '202013024035', '112018008824', 'PI0906951', '102012004079', '102023012754', '102023009444', '102020000955', '112015005351', '112015000808', '102016027868', '102015029049', '102015007934', '112014008059', '112014000676', '102014002048', '102013025651', '122020001985', '122019026850', '122019016068', '112016014536', '112016013519', '112015030352', '112015027282', 'PI1013698', '112015020532', '202020026256', '202020007390', '112021013814', '112021011264', '112015011550', '202016018807', '202013019250', '122022020636', '122022017789', '112018005056', '112013018920', '112012029897', '112012026999', '112012025948', '112012022998', '112012007444', '102020001796', '112015003527', '112015003516', '112014032938', '112014031962', '102016016171', '102016015976', '112014019190', '102015031651', '102015030787', '112014016810', '112014012994', '112014008804', '102014022484', '10

In [13]:
print(len(lista))

84


In [ ]:
# atualiza campo incoerencia com gemini-2.5-flash

import mysql.connector
import time
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
from datetime import date

hoje = date.today()
    
load_dotenv(dotenv_path='.env', override=True)
gemini_api_key=os.getenv("GEMINI_API_KEY")
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=gemini_api_key)

json_data = {"patents": [{"numero": item} for item in lista]}
json_data = json.dumps(json_data, indent=4, ensure_ascii=False)
data = json.loads(json_data)

with open("descricao.sql", "a", encoding="utf-8") as f:
    for patent in data.get("patents", []):
        numero = patent.get("numero")
        if numero != 'NUMERO':
            query_pedido = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM pedido where decisao='indeferimento' and numero='{numero}'" + '"'
            url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query_pedido}"
            print(url)
            try:
                json_data = conectar_siscap(url,return_json=True)
            except:
                print(f"conexão inválida, pulando (indeferimento {numero} não encontrado)...")
                continue

            #print(json_data)
            if not json_data:
                print("json_data vazio, pulando...")
                continue
            try:
                data_pedido = json.loads(json_data)
            except json.JSONDecodeError:
                print("JSON inválido, pulando...")
                continue
    
            if not data_pedido["patents"]:
                print(f"Sem registros para {numero}, pulando...")
                continue
        
            #data_pedido = json.loads(json_data)
            codigo = data_pedido["patents"][0]["codigo"]
            divisao = data_pedido["patents"][0]["divisao"]
        
            #url = f"https://siscap.inpi.gov.br/adm/pareceres/{divisao}/{numero}{codigo}.txt"
            #print(url)
            #texto_relatorio = conectar_siscap(url, return_json=False)
            #print(texto_relatorio)

            url = f"pareceres/{numero}{codigo}.txt"
            caminho_do_arquivo = url
            if os.path.exists(caminho_do_arquivo):
                with open(caminho_do_arquivo, 'r', encoding='utf-8') as arquivo:
                    texto_relatorio = arquivo.read()
            else:
                print(f"Arquivo não encontrado: {caminho_do_arquivo}")
                texto_relatorio = None  # ou "" dependendo do seu uso

            comando = f"SELECT conclusao FROM anterioridades_desc WHERE numero='{numero}'"
            cursor.execute(comando)
            resultado = cursor.fetchall()
            conclusao = resultado[0][0]

            query = f"""Neste parecer verifique se existência de incoerência interna entre as conclusões: [{conclusao}] e o restante do parecer. 
            Ignore os X nos quadros 2 e 3 e considere apenas a discussão que se segue nestes quadros (se houver).
            Quando apenas uma ou mais reivindicação não tem novidade ou atividade inventiva, é justificável indeferir o pedido por falta de novidade 
            ou atividade inventiva, respectivamente. Escreva a saída numa forma corrida, sem bullets, parágrafos, travessões, nem pula linha.
            Apresente uma resposta curta e objetiva. Relatório: {texto_relatorio}"""

            messages=[{"role":"user", "content": query}]
            resumo = ""  # valor padrão caso tudo falhe
            for tentativa in range(5):
                    try:
                        response = llm.invoke(messages)
                        resumo = response.content
                        break
                    except Exception as e:
                        if "429" in str(e):
                            tempo_espera = 2 ** (tentativa+2)
                            print(f"Rate limit atingido. Tentando novamente em {tempo_espera}s...")
                            time.sleep(2 ** tentativa)  # backoff exponencial
                        else:
                            raise
                            
            time.sleep(5)
            #url = "https://api.openai.com/v1/chat/completions"
            #data_json = {
            #    "model": "gpt-5-mini",  # Use o modelo desejado, como 'gpt-5.2' ou 'gpt-4o-mini' ou 'gpt-5-mini'
            #    "messages": [
            #        {"role": "user", "content": query}
            #    ]
            #}
            #headers = {
            #    "Authorization": f"Bearer {openai_api_key}",
            #    "Content-Type": "application/json"
            #}
            #response = requests.post(url, headers=headers, json=data_json, verify=False)
            
            #if response.status_code != 200:
            #    print(f"Não consegui conexão {numero} (HTTP {response.status_code})")
            #    continue
            
            #try:
            #    resposta = response.json()
            #    resumo = resposta["choices"][0]["message"]["content"]
            #except (ValueError, KeyError, IndexError, TypeError):
            #    print(f"Resposta inválida da API para {numero}")
            #    continue

            if resumo:
                texto_corrigido = " ".join(resumo.split())
                texto_corrigido = texto_corrigido.replace("'", "")
                texto_corrigido = texto_corrigido.replace("‑","-")
                sql_resumo = f"UPDATE anterioridades_desc SET incoerencia='{texto_corrigido}', data='{hoje}' WHERE numero='{numero}' and modelo='gemini-2.5-flash';"
                print(sql_resumo)
                f.write(sql_resumo + "\n")
                #break

In [15]:
# faz a atualização de resumo_recurso com gpt-5-nano apenas nos pedidos de abrantes

import mysql.connector
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

comando = f"SELECT * FROM `carga` WHERE examinador='abrantes' and numero in (select numero from arquivados where despacho='12.2' and anulado=0) and numero in (select numero from anterioridades_desc where resumo_recurso='' and modelo='gpt-5-nano')"
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)
lista = df.values.tolist()
if df.shape[1] > 1:
    lista = df.iloc[:, 0].tolist()
else:
    lista = []
lista.insert(0, 'numero')
print(lista)

['numero']


In [80]:
print(len(lista))

1


In [ ]:
# ******************************************UPDATE ANTERIORIDADES_DESC CAMPO RESUMO_RECURSO em gpt-5-nano
# certifique-se de rodar a rotina acima conectar_siscap e de estar a VPN ligada

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts.prompt import PromptTemplate
import PyPDF2
import re

def limpar_caracteres_especiais(texto):
    # Remove caracteres não imprimíveis
    texto = re.sub(r'[^\x20-\x7EÀ-ÿ]', '', texto)
    return texto
    
def format_as_single_paragraph(text):
    # Remove quebras de linha e espaços extras
    formatted_text = ' '.join(line.strip() for line in text.splitlines() if line.strip())
    return formatted_text
    
load_dotenv(dotenv_path='.env', override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")

data["patents"] = lista
#data["patents"] = ['numero','PI1005286','PI1001299'] # lista de numeros especificos

with open("descricao.sql", "a", encoding="utf-8") as f:
    for i in range(1, len(data["patents"])):
        #if i==2: break
        #numero = data["patents"][i]["numero"]
        numero = data["patents"][i] # para ler a lista de numeros especificos
    
        query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM despachos_pag where tipo_peticao='214' and numero='{numero}'" + '"'
        url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
        print(url)
        try:
            json_data = conectar_siscap(url,return_json=True)
            data1 = json.loads(json_data)
            numnossonumero = data1["patents"][0]["numnossonumero"]
            
            texto_relatorio = ''
            url = f"peticoes/{numero}_{numnossonumero}_214.txt"
            caminho_do_arquivo = url
            if os.path.exists(caminho_do_arquivo):
                with open(caminho_do_arquivo, 'r', encoding='utf-8') as arquivo:
                    texto_relatorio = arquivo.read()
            else:
                print(f"Arquivo não encontrado: {caminho_do_arquivo}")
                texto_relatorio = None  # ou "" dependendo do seu uso

            all_text = texto_relatorio
            if all_text!='':
                texto_anon = all_text
                texto_anon = texto_anon.strip()
                if texto_anon:
                    url = "https://api.openai.com/v1/chat/completions"
                    query = f"Resuma os principais argumentos apresentados pela requerente em sua petição recursal {texto_anon}. Formato: Lista de strings (pontos principais) onde cada item segue o darão i), ii), iii) etc.. "
                    #print(query)
                    data_json = {
                        "model": "gpt-5-nano",  
                        "messages": [
                            {"role": "user", "content": query}
                        ]
                    }
                    headers = {
                        "Authorization": f"Bearer {openai_api_key}",
                        "Content-Type": "application/json"
                    }
                    response = requests.post(url, headers=headers, json=data_json, verify=False)
                    if response.status_code == 200:
                        resposta = response.json()
                        resumo = resposta['choices'][0]['message']['content']
                        resumo = resumo.replace("'","")
                        resumo = resumo.replace('"',"")
                        resumo = resumo.replace('“','')
                        resumo = resumo.replace('”','')
                        resumo = format_as_single_paragraph(resumo)
                        sql_resumo = f"UPDATE anterioridades_desc set resumo_recurso='{resumo}' WHERE numero='{numero}' and modelo='gpt-5-nano';"
                        print(sql_resumo)
                        f.write(sql_resumo + "\n")
                    else:
                        print(f"Erro {response.status_code}: {response.text}")
                    
                #if i == 2:
                    #break
        except Exception as e:
            print(f"Não achei petição 214 {numero} {e}")

In [16]:
# faz a atualização de resumo_recurso com qwen/qwen3-32b 

import mysql.connector
conexao = mysql.connector.connect(host='localhost',user='root',password='',database='producao')
cursor = conexao.cursor()

comando = f"SELECT * FROM `carga` WHERE examinador in ('abrantes','milavsl','scaplan','srosa','cidade','eloliveira','apedrosa','rosanab','jsoares','alciclea','szandona','douglasm','cujikawa') and numero in (select numero from arquivados where despacho='12.2' and anulado=0) and numero in (select numero from anterioridades_desc where resumo_recurso='' and modelo='qwen/qwen3-32b')"
cursor.execute(comando)
resultado = cursor.fetchall()
df = pd.DataFrame(resultado)
lista = df.values.tolist()
if df.shape[1] > 1:
    lista = df.iloc[:, 0].tolist()
else:
    lista = []
lista.insert(0, 'numero')
print(lista)

['numero', 'PI0915852', 'PI0908768', '112013020770', '112013019699', '112013018454', '112012017319', '102020005648', '112015005997', '102015006751', '102014031388', '102014023655', '102013024834', '112013029300', '112013026489', '102013000308', '112018001513', 'MU9100240', '122019001140', '122017012058', '112015032570', '112022008919', '112015029135', '112015028880', '112015023151', '202019004832', '202013024035', '122021007541', '122020017521', '112018008824', 'PI0906951', '112015000808', '102016027868', '102015029049', '102015007934', '112014008059', '112014000676', '102014002048', '102013025651', '102013014262', '122020001985', '122019026850', '122019016068', '112016003968', 'PI1013698', '202020026256', '202020007390', '112021023943', '112021011264', '112015011550', '202016018807', '202013019250', '122022020636', '122020017517', '112018005056', '112013019993', '112012029897', '112012026999', '112012025948', '112012022998', '112012007444', '112015002586', '112014032938', '10201503150

In [17]:
print(len(lista))

83


In [ ]:
# ******************************************UPDATE ANTERIORIDADES_DESC CAMPO RESUMO_RECURSO em qwen/qwen3-32b 
# certifique-se de rodar a rotina acima conectar_siscap e de estar a VPN ligada

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts.prompt import PromptTemplate
import PyPDF2
import re
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import re

def clean_answer(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    return text.strip()
    
load_dotenv(dotenv_path='.env', override=True)
groq_api_key = os.getenv("GROQ_API_KEY")

def limpar_caracteres_especiais(texto):
    # Remove caracteres não imprimíveis
    texto = re.sub(r'[^\x20-\x7EÀ-ÿ]', '', texto)
    return texto
    
def format_as_single_paragraph(text):
    # Remove quebras de linha e espaços extras
    formatted_text = ' '.join(line.strip() for line in text.splitlines() if line.strip())
    return formatted_text
    
load_dotenv(dotenv_path='.env', override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")

data["patents"] = lista
#data["patents"] = ['numero','PI1005286','PI1001299'] # lista de numeros especificos

with open("descricao.sql", "a", encoding="utf-8") as f:
    for i in range(1, len(data["patents"])):
        #if i==2: break
        #numero = data["patents"][i]["numero"]
        numero = data["patents"][i] # para ler a lista de numeros especificos
    
        query = '"' + "mysql_query" + '"' ":" + '"' + f" * FROM despachos_pag where tipo_peticao='214' and numero='{numero}'" + '"'
        url = f"https://cientistaspatentes.com.br/apiphp/patents/query/?q={query}"
        print(url)
        try:
            json_data = conectar_siscap(url,return_json=True)
            data1 = json.loads(json_data)
            numnossonumero = data1["patents"][0]["numnossonumero"]
            
            texto_relatorio = ''
            url = f"peticoes/{numero}_{numnossonumero}_214.txt"
            caminho_do_arquivo = url
            if os.path.exists(caminho_do_arquivo):
                with open(caminho_do_arquivo, 'r', encoding='utf-8') as arquivo:
                    texto_relatorio = arquivo.read()
            else:
                print(f"Arquivo não encontrado: {caminho_do_arquivo}")
                texto_relatorio = None  # ou "" dependendo do seu uso

            all_text = texto_relatorio
            if all_text!='':
                texto_anon = all_text
                texto_anon = texto_anon.strip()
                if texto_anon:
                    query = f"Resuma os principais argumentos apresentados pela requerente em sua petição recursal {texto_anon}. Formato: Lista de strings (pontos principais) onde cada item segue o padrão i), ii), iii) etc.. "
                    MAX_CHARS = 12000  # ajuste conforme necessário
                    if len(query) > MAX_CHARS:
                        query = query[:MAX_CHARS]
                        
                    llm = ChatGroq(model="qwen/qwen3-32b")
                    messages=[{"role":"user", "content": query}]
                    response = llm.invoke(messages)
                    resumo = clean_answer(response.content)
                    time.sleep(3) 

                    resumo = " ".join(resumo.split())
                    resumo = resumo.replace("'", "")
                    resumo = resumo.replace("‑","-")
                    resumo = resumo.replace("'","")
                    resumo = resumo.replace('"',"")
                    resumo = resumo.replace('“','')
                    resumo = resumo.replace('”','')
                    resumo = format_as_single_paragraph(resumo)
                    sql_resumo = f"UPDATE anterioridades_desc set resumo_recurso='{resumo}' WHERE numero='{numero}' and modelo='qwen/qwen3-32b';"
                    print(sql_resumo)
                    f.write(sql_resumo + "\n")

        except Exception as e:
            print(f"Não achei petição 214 {numero} {e}")